# O "programa dos sonhos" da FidelidadeScript

Antes de definir o analisador ou as demais fases do compilador, descrevemos o programa que queremos permitir que uma pessoa escreva. O exemplo combina uma compra, o cálculo do cashback, a concessão de pontos e operações comuns de uma conta de fidelidade.

Este programa é o contrato inicial da linguagem: as próximas etapas deverão torná-lo executável e validar as regras de negócio do domínio.

A primeira célula executável valida a ficha antes de definir o programa. Se os requisitos mínimos não forem atendidos, ela lança um erro.

In [1]:
# ============================================================
# FICHA DE QUALIDADE — validar antes de definir o programa DSL
# ============================================================

import re

MINHA_FICHA = {
    "nome": "FidelidadeScript",
    "dominio": "programas de pontos e cashback",
    "tipos": ["inteiro", "decimal", "texto", "logico"],
    "comandos_dominio": [
        'creditar "<cliente>", <pontos>;',
        'resgatar "<cliente>", "<cupom>", <pontos>;',
        'expirar "<cliente>", <pontos>;',
        'extrato "<cliente>";',
    ],
    "regras_de_negocio": [
        "percentual de cashback acima de 100% gera erro",
        "pontos negativos geram erro",
        "resgate acima do saldo disponível gera erro de execução",
    ],
    "programas_exemplo": [
        r'''
decimal compra = 250.0;
decimal percentual_cashback = 10.0;
decimal cashback = compra * percentual_cashback / 100.0;
inteiro pontos = 100;
se compra >= 200.0 {
    pontos = pontos + 50;
}
creditar "cliente_42", pontos;
extrato "cliente_42";
''',
        r'''
decimal percentual_cashback = 12.0;
inteiro pontos = 5;
se percentual_cashback <= 100.0 {
    creditar "cliente_42", pontos;
}
extrato "cliente_42";
''',
        r'''
inteiro pontos = 150;
creditar "cliente_42", pontos;
se pontos >= 100 {
    resgatar "cliente_42", "cupom_10", 100;
}
extrato "cliente_42";
''',
    ],
}

def conferir_ficha(ficha):
    if not isinstance(ficha, dict):
        raise TypeError("A ficha da linguagem deve ser um dicionário.")

    def tem_texto(valor):
        return isinstance(valor, str) and bool(valor.strip())

    def quantidade_itens(campo):
        itens = ficha.get(campo)
        if not isinstance(itens, list):
            return 0
        return sum(tem_texto(item) for item in itens)

    nome = ficha.get("nome")
    dominio = ficha.get("dominio")
    identidade_valida = (
        tem_texto(nome)
        and tem_texto(dominio)
        and "todo" not in (nome + dominio).lower()
    )
    programas = ficha.get("programas_exemplo")
    tres_programas_validos = (
        isinstance(programas, list)
        and len(programas) == 3
        and all(tem_texto(programa) for programa in programas)
    )
    todos_usam_controle = (
        tres_programas_validos
        and all(
            re.search(r"(?m)^\s*(?:se|enquanto)\b", programa)
            for programa in programas
        )
    )
    checagens = [
        ("nome e domínio preenchidos", identidade_valida),
        ("pelo menos 3 tipos", quantidade_itens("tipos") >= 3),
        ("pelo menos 2 comandos do domínio", quantidade_itens("comandos_dominio") >= 2),
        ("pelo menos 2 regras de negócio", quantidade_itens("regras_de_negocio") >= 2),
        ("exatamente 3 programas-exemplo", tres_programas_validos),
        ("cada programa usa se ou enquanto", todos_usam_controle),
    ]

    for texto, ok in checagens:
        print(("✅" if ok else "⬜"), texto)

    faltam = [texto for texto, ok in checagens if not ok]
    if faltam:
        raise ValueError("Ficha da linguagem incompleta: " + "; ".join(faltam))

    print("\n🚀 Ficha completa! O programa de exemplo pode ser carregado.")


conferir_ficha(MINHA_FICHA)

# ============================================================
# PASSO 0 — O "programa dos sonhos" da FidelidadeScript
# ============================================================

PROGRAMA_FIDELIDADE = r'''
// FidelidadeScript — campanha de cashback e pontos
decimal compra = 250.0;
decimal percentual_cashback = 10.0;
decimal cashback = compra * percentual_cashback / 100.0;
inteiro pontos = 100;

se compra >= 200.0 {
    pontos = pontos + 50;
}

mostrar "Cashback calculado:";
mostrar cashback;
creditar "cliente_42", pontos;
extrato "cliente_42";
resgatar "cliente_42", "cupom_10", 100;
expirar "cliente_42", 25;
extrato "cliente_42";
'''

print("Programa com", PROGRAMA_FIDELIDADE.count("\n") - 1, "linhas carregado em PROGRAMA_FIDELIDADE.")

✅ nome e domínio preenchidos
✅ pelo menos 3 tipos
✅ pelo menos 2 comandos do domínio
✅ pelo menos 2 regras de negócio
✅ exatamente 3 programas-exemplo
✅ cada programa usa se ou enquanto

🚀 Ficha completa! O programa de exemplo pode ser carregado.
Programa com 17 linhas carregado em PROGRAMA_FIDELIDADE.


## Passo 1 — Análise léxica

Nesta etapa, o Lark reconhece os tipos, as palavras reservadas e os símbolos da FidelidadeScript. Comentários e espaços são ignorados; erros léxicos informam localização e dica.

In [2]:
# ============================================================
# PASSO 1 — Análise léxica da FidelidadeScript
# ============================================================

import re
from dataclasses import dataclass

from lark import Lark
from lark.exceptions import UnexpectedCharacters

TERMINAIS = r'''
// ---------- palavras reservadas e tipos ----------
T_INTEIRO: "inteiro"
T_DECIMAL: "decimal"
T_TEXTO: "texto"
T_LOGICO: "logico"
MOSTRAR: "mostrar"
CREDITAR: "creditar"
RESGATAR: "resgatar"
EXPIRAR: "expirar"
EXTRATO: "extrato"
SE: "se"
SENAO: "senao"
ENQUANTO: "enquanto"
E: "e"
OU: "ou"
NAO: "nao"
VERDADEIRO: "verdadeiro"
FALSO: "falso"

// ---------- literais e identificadores ----------
NUM_DECIMAL.2: /[0-9]+\.[0-9]+/
NUM_INTEIRO: /[0-9]+/
TEXTO_LIT: /"[^"\n]*"/
NOME: /(?!(?:inteiro|decimal|texto|logico|mostrar|creditar|resgatar|expirar|extrato|se|senao|enquanto|e|ou|nao|verdadeiro|falso)\b)[a-zA-Z][a-zA-Z0-9_]*/

// ---------- operadores ----------
MAIOR_IGUAL: ">="
MENOR_IGUAL: "<="
IGUAL_IGUAL: "=="
DIFERENTE: "!="
MAIOR: ">"
MENOR: "<"
ATRIB: "="
MAIS: "+"
MENOS: "-"
VEZES: "*"
DIVIDE: "/"

// ---------- pontuação ----------
PONTO_VIRGULA: ";"
VIRGULA: ","
ABRE_CHAVE: "{"
FECHA_CHAVE: "}"
ABRE_PAR: "("
FECHA_PAR: ")"

// ---------- elementos ignorados ----------
COMENTARIO: /\/\/[^\n]*/
%import common.WS
%ignore WS
%ignore COMENTARIO
'''

PALAVRAS_RESERVADAS = {
    "inteiro": "T_INTEIRO",
    "decimal": "T_DECIMAL",
    "texto": "T_TEXTO",
    "logico": "T_LOGICO",
    "mostrar": "MOSTRAR",
    "creditar": "CREDITAR",
    "resgatar": "RESGATAR",
    "expirar": "EXPIRAR",
    "extrato": "EXTRATO",
    "se": "SE",
    "senao": "SENAO",
    "enquanto": "ENQUANTO",
    "e": "E",
    "ou": "OU",
    "nao": "NAO",
    "verdadeiro": "VERDADEIRO",
    "falso": "FALSO",
}

NOMES_TERMINAIS = [
    nome
    for nome in re.findall(r"^([A-Z_]+)(?:\.\d+)?:", TERMINAIS, re.MULTILINE)
    if nome != "COMENTARIO"
]

if len(NOMES_TERMINAIS) < 20:
    raise RuntimeError("A FidelidadeScript precisa definir pelo menos 20 tipos de token.")
if len(PALAVRAS_RESERVADAS) < 8 or not {"creditar", "resgatar", "expirar", "extrato"}.issubset(PALAVRAS_RESERVADAS):
    raise RuntimeError("Faltam palavras reservadas ou comandos do domínio no léxico.")

_lexer_lark = Lark(
    "start: token*\ntoken: " + " | ".join(NOMES_TERMINAIS) + "\n" + TERMINAIS,
    parser="lalr",
    lexer="basic",
)

@dataclass
class Diagnostico:
    fase: str
    mensagem: str
    linha: int
    coluna: int
    dica: str = ""
    nivel: str = "erro"

    def __str__(self):
        local = f"linha {self.linha}, coluna {self.coluna}"
        texto = f"[{self.fase}] {local}: {self.mensagem}"
        prefixo = "AVISO " if self.nivel == "aviso" else ""
        return prefixo + texto + (f"\nDica: {self.dica}" if self.dica else "")

class ErroCompilacao(Exception):
    def __init__(self, diagnosticos):
        self.diagnosticos = diagnosticos
        super().__init__("\n".join(str(item) for item in diagnosticos))

DICAS_LEXICAS = {
    "'": 'Textos usam aspas duplas, por exemplo: "cliente_42".',
    '"': "Literal de texto sem fechamento; feche as aspas na mesma linha.",
    "&": "Para conjunção lógica, use a palavra reservada 'e'.",
    "|": "Para disjunção lógica, use a palavra reservada 'ou'.",
    "!": "Para negação, use 'nao'; para diferença, use '!='.",
    "#": "Comentários começam com //.",
    "%": "Calcule cashback com percentual / 100; '%' não é um operador da linguagem.",
}

def dica_lexica(caractere):
    if caractere in DICAS_LEXICAS:
        return DICAS_LEXICAS[caractere]
    if caractere.isalpha():
        return "Identificadores usam letras sem acento; por exemplo, escreva 'compra'."
    return "Remova o símbolo ou use um operador/pontuação definido pela FidelidadeScript."

def tokenizar(fonte):
    try:
        return list(_lexer_lark.lex(fonte))
    except UnexpectedCharacters as erro:
        caractere = erro.char
        diagnostico = Diagnostico(
            "léxico",
            f"caractere inesperado {caractere!r}",
            erro.line,
            erro.column,
            dica_lexica(caractere),
        )
        raise ErroCompilacao([diagnostico]) from None

def mostrar_tokens(fonte, limite=None):
    tokens = tokenizar(fonte)
    exibidos = tokens if limite is None else tokens[:limite]
    print(f"{'#':>3}  {'TIPO':<16} {'LEXEMA':<24} LINHA:COLUNA")
    print("-" * 62)
    for indice, token in enumerate(exibidos, 1):
        print(f"{indice:>3}  {token.type:<16} {token.value:<24} {token.line}:{token.column}")
    return tokens

print(f"Tipos de token: {len(NOMES_TERMINAIS)} | Palavras reservadas: {len(PALAVRAS_RESERVADAS)}")
print("Tabela de tokens de PROGRAMA_FIDELIDADE:")
TOKENS_PROGRAMA = mostrar_tokens(PROGRAMA_FIDELIDADE)

ERROS_LEXICOS_DEMONSTRACAO = [
    ("aspas simples", "texto cliente = 'cliente_42';"),
    ("operador lógico inválido", "se pontos > 0 & compra > 0 {}"),
    ("identificador com acento", "inteiro prêmio = 10;"),
]

for nome_caso, fonte_errada in ERROS_LEXICOS_DEMONSTRACAO:
    try:
        tokenizar(fonte_errada)
    except ErroCompilacao as erro:
        print(f"\nExemplo de erro ({nome_caso}):\n{erro}")
    else:
        raise AssertionError(f"O exemplo de erro lexical '{nome_caso}' deveria falhar.")

Tipos de token: 38 | Palavras reservadas: 17
Tabela de tokens de PROGRAMA_FIDELIDADE:
  #  TIPO             LEXEMA                   LINHA:COLUNA
--------------------------------------------------------------
  1  T_DECIMAL        decimal                  3:1
  2  NOME             compra                   3:9
  3  ATRIB            =                        3:16
  4  NUM_DECIMAL      250.0                    3:18
  5  PONTO_VIRGULA    ;                        3:23
  6  T_DECIMAL        decimal                  4:1
  7  NOME             percentual_cashback      4:9
  8  ATRIB            =                        4:29
  9  NUM_DECIMAL      10.0                     4:31
 10  PONTO_VIRGULA    ;                        4:35
 11  T_DECIMAL        decimal                  5:1
 12  NOME             cashback                 5:9
 13  ATRIB            =                        5:18
 14  NOME             compra                   5:20
 15  VEZES            *                        5:27
 16  NOME        

## Passo 2 — Análise sintática e AST

A gramática LALR organiza declarações, comandos e expressões em níveis de precedência. O Transformer converte a árvore sintática em nós `dataclass`; erros indicam a posição e os símbolos esperados.

In [3]:
# ============================================================
# PASSO 2 — Análise sintática e construção da AST
# ============================================================

from dataclasses import dataclass, fields, is_dataclass
from lark import Lark, Transformer
from lark.exceptions import UnexpectedInput

REGRAS = r'''
start: programa
programa: comando*

?comando: declaracao
        | atribuicao
        | mostrar_stmt
        | se_stmt
        | enquanto_stmt
        | creditar_stmt
        | resgatar_stmt
        | expirar_stmt
        | extrato_stmt

declaracao: tipo NOME (ATRIB expr)? PONTO_VIRGULA
tipo: T_INTEIRO -> tipo_inteiro
    | T_DECIMAL -> tipo_decimal
    | T_TEXTO -> tipo_texto
    | T_LOGICO -> tipo_logico
atribuicao: NOME ATRIB expr PONTO_VIRGULA
mostrar_stmt: MOSTRAR expr PONTO_VIRGULA
se_stmt: SE expr bloco (SENAO bloco)?
enquanto_stmt: ENQUANTO expr bloco
bloco: ABRE_CHAVE comando* FECHA_CHAVE

creditar_stmt: CREDITAR expr VIRGULA expr PONTO_VIRGULA
resgatar_stmt: RESGATAR expr VIRGULA expr VIRGULA expr PONTO_VIRGULA
expirar_stmt: EXPIRAR expr VIRGULA expr PONTO_VIRGULA
extrato_stmt: EXTRATO expr PONTO_VIRGULA

// Precedência, da menor para a maior: ou, e, comparações, soma, produto e unários.
expr: logica_ou
logica_ou: logica_e (OU logica_e)*
logica_e: logica_nao (E logica_nao)*
logica_nao: NAO logica_nao -> unop
          | comparacao
comparacao: soma_expr ((MAIOR | MENOR | MAIOR_IGUAL | MENOR_IGUAL | IGUAL_IGUAL | DIFERENTE) soma_expr)*
soma_expr: produto_expr ((MAIS | MENOS) produto_expr)*
produto_expr: fator ((VEZES | DIVIDE) fator)*
fator: MENOS fator -> unop
     | MAIS fator -> unop
     | atom
atom: NUM_DECIMAL -> lit_decimal
    | NUM_INTEIRO -> lit_inteiro
    | TEXTO_LIT -> lit_texto
    | VERDADEIRO -> lit_logico
    | FALSO -> lit_logico
    | NOME -> var
    | ABRE_PAR expr FECHA_PAR -> agrupamento
'''

GRAMATICA = REGRAS + TERMINAIS
parser = Lark(GRAMATICA, parser="lalr", lexer="basic")

@dataclass
class Expr:
    linha: int

@dataclass
class Lit(Expr):
    valor: object
    tipo: str = None

@dataclass
class Var(Expr):
    nome: str
    tipo: str = None

@dataclass
class BinOp(Expr):
    op: str
    esq: Expr
    dir: Expr
    tipo: str = None

@dataclass
class UnOp(Expr):
    op: str
    expr: Expr
    tipo: str = None

@dataclass
class Decl:
    linha: int
    tipo: str
    nome: str
    expr: object = None

@dataclass
class Atrib:
    linha: int
    nome: str
    expr: Expr

@dataclass
class Mostrar:
    linha: int
    expr: Expr

@dataclass
class Se:
    linha: int
    cond: Expr
    entao: list
    senao: object = None

@dataclass
class Enquanto:
    linha: int
    cond: Expr
    corpo: list

@dataclass
class Creditar:
    linha: int
    cliente: Expr
    pontos: Expr

@dataclass
class Resgatar:
    linha: int
    cliente: Expr
    cupom: Expr
    pontos: Expr

@dataclass
class Expirar:
    linha: int
    cliente: Expr
    pontos: Expr

@dataclass
class Extrato:
    linha: int
    cliente: Expr

@dataclass
class Programa:
    comandos: list

class MontaAST(Transformer):
    def start(self, filhos):
        return filhos[0]

    def programa(self, filhos):
        return Programa(list(filhos))

    def bloco(self, filhos):
        return list(filhos[1:-1])

    def tipo_inteiro(self, filhos):
        return "inteiro"

    def tipo_decimal(self, filhos):
        return "decimal"

    def tipo_texto(self, filhos):
        return "texto"

    def tipo_logico(self, filhos):
        return "logico"

    def declaracao(self, filhos):
        tipo, nome = filhos[:2]
        expr = filhos[3] if len(filhos) == 5 else None
        return Decl(nome.line, tipo, str(nome), expr)

    def atribuicao(self, filhos):
        nome, _, expr, _ = filhos
        return Atrib(nome.line, str(nome), expr)

    def mostrar_stmt(self, filhos):
        palavra, expr, _ = filhos
        return Mostrar(palavra.line, expr)

    def se_stmt(self, filhos):
        palavra, cond, entao = filhos[:3]
        senao = filhos[4] if len(filhos) == 5 else None
        return Se(palavra.line, cond, entao, senao)

    def enquanto_stmt(self, filhos):
        palavra, cond, corpo = filhos
        return Enquanto(palavra.line, cond, corpo)

    def creditar_stmt(self, filhos):
        palavra, cliente, _, pontos, _ = filhos
        return Creditar(palavra.line, cliente, pontos)

    def resgatar_stmt(self, filhos):
        palavra, cliente, _, cupom, _, pontos, _ = filhos
        return Resgatar(palavra.line, cliente, cupom, pontos)

    def expirar_stmt(self, filhos):
        palavra, cliente, _, pontos, _ = filhos
        return Expirar(palavra.line, cliente, pontos)

    def extrato_stmt(self, filhos):
        palavra, cliente, _ = filhos
        return Extrato(palavra.line, cliente)

    def expr(self, filhos):
        return filhos[0]

    def logica_nao(self, filhos):
        return filhos[0]

    def _binaria(self, filhos):
        resultado = filhos[0]
        for indice in range(1, len(filhos), 2):
            operador, direita = filhos[indice], filhos[indice + 1]
            resultado = BinOp(operador.line, str(operador), resultado, direita)
        return resultado

    def logica_ou(self, filhos):
        return self._binaria(filhos)

    def logica_e(self, filhos):
        return self._binaria(filhos)

    def comparacao(self, filhos):
        return self._binaria(filhos)

    def soma_expr(self, filhos):
        return self._binaria(filhos)

    def produto_expr(self, filhos):
        return self._binaria(filhos)

    def unop(self, filhos):
        operador, expr = filhos
        return UnOp(operador.line, str(operador), expr)

    def fator(self, filhos):
        return filhos[0]

    def lit_decimal(self, filhos):
        token = filhos[0]
        return Lit(token.line, float(token), "decimal")

    def lit_inteiro(self, filhos):
        token = filhos[0]
        return Lit(token.line, int(token), "inteiro")

    def lit_texto(self, filhos):
        token = filhos[0]
        return Lit(token.line, str(token)[1:-1], "texto")

    def lit_logico(self, filhos):
        token = filhos[0]
        return Lit(token.line, token.type == "VERDADEIRO", "logico")

    def var(self, filhos):
        token = filhos[0]
        return Var(token.line, str(token))

    def agrupamento(self, filhos):
        return filhos[1]

def terminal_legivel(nome):
    nomes_amigaveis = {
        "NOME": "um identificador",
        "NUM_INTEIRO": "um número inteiro",
        "NUM_DECIMAL": "um número decimal",
        "TEXTO_LIT": "um texto entre aspas",
    }
    if nome in nomes_amigaveis:
        return nomes_amigaveis[nome]
    try:
        padrao = parser.get_terminal(nome).pattern
    except KeyError:
        return nome
    return repr(padrao.value) if padrao.type == "str" else nome

def analisar_sintaxe(fonte):
    tokenizar(fonte)
    try:
        arvore = parser.parse(fonte)
    except UnexpectedInput as erro:
        prioridade = {"PONTO_VIRGULA": 0, "VIRGULA": 1, "FECHA_CHAVE": 2, "FECHA_PAR": 3, "ATRIB": 4}
        terminais_esperados = set(getattr(erro, "expected", ()) or ())
        esperado = sorted(
            terminais_esperados,
            key=lambda nome: (prioridade.get(nome, 99), terminal_legivel(nome)),
        )
        lista_esperada = ", ".join(terminal_legivel(item) for item in esperado[:6])
        if not lista_esperada:
            lista_esperada = "uma continuação válida"
        token = getattr(erro, "token", None)
        fim_inesperado = token is None or getattr(token, "type", None) == "$END"
        linha = getattr(erro, "line", None) or fonte.count("\n") + 1
        coluna = getattr(erro, "column", None) or len(fonte.rsplit("\n", 1)[-1]) + 1
        mensagem = "fim inesperado do programa" if fim_inesperado else f"token inesperado {token.value!r}"
        dica = f"Esperado: {lista_esperada}"
        raise ErroCompilacao([Diagnostico("sintático", mensagem, linha, coluna, dica)]) from None
    return MontaAST().transform(arvore)

def mostrar_ast(no, nivel=0):
    prefixo = "  " * nivel
    if isinstance(no, list):
        print(f"{prefixo}[{len(no)} item(ns)]")
        for filho in no:
            mostrar_ast(filho, nivel + 1)
    elif is_dataclass(no):
        linha = f" (linha {no.linha})" if hasattr(no, "linha") else ""
        print(f"{prefixo}{type(no).__name__}{linha}")
        for campo in fields(no):
            if campo.name == "linha":
                continue
            print(f"{prefixo}  {campo.name}:")
            mostrar_ast(getattr(no, campo.name), nivel + 2)
    else:
        print(f"{prefixo}{no!r}")

PROGRAMA_SINTATICO = r'''
inteiro pontos = 100;
decimal compra = 250.0;
se compra >= 200.0 e pontos > 0 {
    creditar "cliente_42", pontos;
} senao {
    extrato "cliente_42";
}
enquanto pontos > 0 {
    expirar "cliente_42", 1;
    pontos = pontos - 1;
}
resgatar "cliente_42", "cupom_10", 50;
'''

AST_FIDELIDADE = analisar_sintaxe(PROGRAMA_SINTATICO)
print("AST do programa FidelidadeScript:")
mostrar_ast(AST_FIDELIDADE)

EXEMPLO_PRECEDENCIA = "mostrar nao falso ou verdadeiro e 2 + 3 * 4 >= 10;"
print("\nAST para demonstrar precedência lógica, relacional e aritmética:")
mostrar_ast(analisar_sintaxe(EXEMPLO_PRECEDENCIA))

ERROS_SINTATICOS_DEMONSTRACAO = [
    "inteiro pontos = 50\nextrato \"cliente_42\";",
    "se verdadeiro { mostrar 1;",
]
for fonte_errada in ERROS_SINTATICOS_DEMONSTRACAO:
    try:
        analisar_sintaxe(fonte_errada)
    except ErroCompilacao as erro:
        print(f"\nExemplo de erro sintático:\n{erro}")
    else:
        raise AssertionError("O exemplo sintaticamente inválido deveria falhar.")

AST do programa FidelidadeScript:
Programa
  comandos:
    [5 item(ns)]
      Decl (linha 2)
        tipo:
          'inteiro'
        nome:
          'pontos'
        expr:
          Lit (linha 2)
            valor:
              100
            tipo:
              'inteiro'
      Decl (linha 3)
        tipo:
          'decimal'
        nome:
          'compra'
        expr:
          Lit (linha 3)
            valor:
              250.0
            tipo:
              'decimal'
      Se (linha 4)
        cond:
          BinOp (linha 4)
            op:
              'e'
            esq:
              BinOp (linha 4)
                op:
                  '>='
                esq:
                  Var (linha 4)
                    nome:
                      'compra'
                    tipo:
                      None
                dir:
                  Lit (linha 4)
                    valor:
                      200.0
                    tipo:
                      'decimal'
    

## Passo 3 — Análise semântica

A análise semântica verifica declarações, escopos de bloco, tipos e regras da FidelidadeScript. Erros impedem que o programa seja considerado válido; avisos informam situações sem bloquear a análise.

In [4]:
# ============================================================
# PASSO 3 — Análise semântica da FidelidadeScript
# ============================================================

@dataclass
class Conv(Expr):
    expr: Expr
    tipo: str = "decimal"

class TabelaSimbolos:
    def __init__(self):
        self.escopos = [{}]

    def abrir(self):
        self.escopos.append({})

    def fechar(self):
        if len(self.escopos) == 1:
            raise RuntimeError("Não é possível fechar o escopo global.")
        return self.escopos.pop()

    def declarar(self, nome, tipo, linha, constante=None):
        atual = self.escopos[-1]
        if nome in atual:
            return False
        atual[nome] = {
            "tipo": tipo,
            "linha": linha,
            "usada": False,
            "constante": constante,
        }
        return True

    def buscar(self, nome):
        for escopo in reversed(self.escopos):
            if nome in escopo:
                return escopo[nome]
        return None

class AnalisadorSemantico:
    NUMERICOS = {"inteiro", "decimal"}
    RELACIONAIS = {">", "<", ">=", "<=", "==", "!="}

    def __init__(self):
        self.tabela = TabelaSimbolos()
        self.diagnosticos = []

    @property
    def erros(self):
        return [d for d in self.diagnosticos if d.nivel == "erro"]

    @property
    def avisos(self):
        return [d for d in self.diagnosticos if d.nivel == "aviso"]

    def erro(self, linha, mensagem, dica=""):
        self.diagnosticos.append(Diagnostico("semântico", mensagem, linha, 0, dica))

    def aviso(self, linha, mensagem, dica=""):
        self.diagnosticos.append(
            Diagnostico("semântico", mensagem, linha, 0, dica, nivel="aviso")
        )

    @staticmethod
    def eh_numero(valor):
        return isinstance(valor, (int, float)) and not isinstance(valor, bool)

    def valor_constante(self, expr):
        if isinstance(expr, Lit):
            return expr.valor
        if isinstance(expr, Conv):
            valor = self.valor_constante(expr.expr)
            return None if valor is None else float(valor)
        if isinstance(expr, Var):
            info = self.tabela.buscar(expr.nome)
            return None if info is None else info["constante"]
        if isinstance(expr, UnOp):
            valor = self.valor_constante(expr.expr)
            if valor is None:
                return None
            if expr.op == "-" and self.eh_numero(valor):
                return -valor
            if expr.op == "+" and self.eh_numero(valor):
                return valor
            if expr.op == "nao" and isinstance(valor, bool):
                return not valor
            return None
        if isinstance(expr, BinOp):
            esquerda = self.valor_constante(expr.esq)
            direita = self.valor_constante(expr.dir)
            if esquerda is None or direita is None:
                return None
            if expr.op == "+":
                if self.eh_numero(esquerda) and self.eh_numero(direita):
                    return esquerda + direita
                if isinstance(esquerda, str) and isinstance(direita, str):
                    return esquerda + direita
                return None
            if expr.op in ("-", "*", "/"):
                if not self.eh_numero(esquerda) or not self.eh_numero(direita):
                    return None
                if expr.op == "-":
                    return esquerda - direita
                if expr.op == "*":
                    return esquerda * direita
                if direita != 0:
                    return esquerda / direita
                return None
            if expr.op == "e" and isinstance(esquerda, bool) and isinstance(direita, bool):
                return esquerda and direita
            if expr.op == "ou" and isinstance(esquerda, bool) and isinstance(direita, bool):
                return esquerda or direita
            if expr.op == "==":
                return esquerda == direita
            if expr.op == "!=":
                return esquerda != direita
            if self.eh_numero(esquerda) and self.eh_numero(direita):
                if expr.op == ">":
                    return esquerda > direita
                if expr.op == "<":
                    return esquerda < direita
                if expr.op == ">=":
                    return esquerda >= direita
                if expr.op == "<=":
                    return esquerda <= direita
        return None

    def converter(self, expr, tipo_destino, linha, contexto):
        if expr.tipo in (tipo_destino, None):
            return expr
        if expr.tipo == "inteiro" and tipo_destino == "decimal":
            return Conv(expr.linha, expr)
        dica = "Use decimal quando o valor tiver casas decimais." if (expr.tipo, tipo_destino) == ("decimal", "inteiro") else ""
        self.erro(linha, f"{contexto}: esperava {tipo_destino}, recebeu {expr.tipo}", dica)
        return expr

    def conferir_percentual(self, nome, expr, linha):
        if nome != "percentual_cashback":
            return
        if expr is None:
            return
        if expr.tipo is not None and expr.tipo not in self.NUMERICOS:
            self.erro(linha, "percentual de cashback precisa ser inteiro ou decimal")
            return
        valor = self.valor_constante(expr)
        if self.eh_numero(valor) and valor > 100:
            self.erro(linha, f"percentual de cashback inválido ({valor}%)", "o percentual máximo é 100%")
        elif self.eh_numero(valor) and valor == 0:
            self.aviso(linha, "percentual de cashback igual a 0%", "a campanha não concederá cashback")

    def analisar(self, programa):
        self.bloco(programa.comandos, novo_escopo=False)
        self.checar_nao_usadas(self.tabela.escopos[0])
        return programa

    def bloco(self, comandos, novo_escopo=True):
        if novo_escopo:
            self.tabela.abrir()
        for comando in comandos:
            self.comando(comando)
        if novo_escopo:
            self.checar_nao_usadas(self.tabela.fechar())

    def checar_nao_usadas(self, escopo):
        for nome, info in escopo.items():
            if not info["usada"]:
                self.aviso(info["linha"], f"'{nome}' foi declarada e nunca usada")

    def conferir_pontos(self, expr, linha, contexto):
        expr = self.converter(self.expr(expr), "inteiro", linha, contexto)
        valor = self.valor_constante(expr)
        if self.eh_numero(valor) and valor < 0:
            self.erro(linha, f"quantidade de pontos negativa ({valor})", "pontos não podem ser negativos")
        elif self.eh_numero(valor) and valor == 0 and contexto == "creditar":
            self.aviso(linha, "crédito de zero pontos", "a operação não altera o saldo do cliente")
        return expr

    def condicao(self, expr, linha, comando):
        expr = self.expr(expr)
        if expr.tipo not in ("logico", None):
            self.erro(linha, f"a condição de '{comando}' precisa ser logico, mas é {expr.tipo}")
        return expr

    def comando(self, comando):
        if isinstance(comando, Decl):
            if comando.expr is not None:
                comando.expr = self.converter(
                    self.expr(comando.expr), comando.tipo, comando.linha,
                    f"declaração de '{comando.nome}'",
                )
            constante = self.valor_constante(comando.expr) if comando.expr is not None else None
            if not self.tabela.declarar(comando.nome, comando.tipo, comando.linha, constante):
                self.erro(comando.linha, f"'{comando.nome}' já foi declarada neste bloco")
            self.conferir_percentual(comando.nome, comando.expr, comando.linha)
        elif isinstance(comando, Atrib):
            info = self.tabela.buscar(comando.nome)
            comando.expr = self.expr(comando.expr)
            if info is None:
                self.erro(comando.linha, f"'{comando.nome}' não foi declarada")
            else:
                comando.expr = self.converter(comando.expr, info["tipo"], comando.linha, f"atribuição a '{comando.nome}'")
                info["constante"] = self.valor_constante(comando.expr)
                self.conferir_percentual(comando.nome, comando.expr, comando.linha)
        elif isinstance(comando, Mostrar):
            comando.expr = self.expr(comando.expr)
        elif isinstance(comando, Se):
            comando.cond = self.condicao(comando.cond, comando.linha, "se")
            self.bloco(comando.entao)
            if comando.senao is not None:
                self.bloco(comando.senao)
        elif isinstance(comando, Enquanto):
            comando.cond = self.condicao(comando.cond, comando.linha, "enquanto")
            self.bloco(comando.corpo)
        elif isinstance(comando, Creditar):
            comando.cliente = self.converter(self.expr(comando.cliente), "texto", comando.linha, "creditar (cliente)")
            comando.pontos = self.conferir_pontos(comando.pontos, comando.linha, "creditar")
        elif isinstance(comando, Resgatar):
            comando.cliente = self.converter(self.expr(comando.cliente), "texto", comando.linha, "resgatar (cliente)")
            comando.cupom = self.converter(self.expr(comando.cupom), "texto", comando.linha, "resgatar (cupom)")
            comando.pontos = self.conferir_pontos(comando.pontos, comando.linha, "resgatar")
        elif isinstance(comando, Expirar):
            comando.cliente = self.converter(self.expr(comando.cliente), "texto", comando.linha, "expirar (cliente)")
            comando.pontos = self.conferir_pontos(comando.pontos, comando.linha, "expirar")
        elif isinstance(comando, Extrato):
            comando.cliente = self.converter(self.expr(comando.cliente), "texto", comando.linha, "extrato (cliente)")
        else:
            raise TypeError(f"Comando AST não suportado: {type(comando).__name__}")

    def expr(self, expr):
        if isinstance(expr, Lit):
            return expr
        if isinstance(expr, Conv):
            expr.expr = self.expr(expr.expr)
            return expr
        if isinstance(expr, Var):
            info = self.tabela.buscar(expr.nome)
            if info is None:
                self.erro(expr.linha, f"'{expr.nome}' não foi declarada", "declare a variável antes de usá-la")
            else:
                info["usada"] = True
                expr.tipo = info["tipo"]
            return expr
        if isinstance(expr, UnOp):
            expr.expr = self.expr(expr.expr)
            tipo = expr.expr.tipo
            if expr.op == "nao":
                if tipo not in ("logico", None):
                    self.erro(expr.linha, f"'nao' exige logico, recebeu {tipo}")
                expr.tipo = "logico"
            else:
                if tipo not in self.NUMERICOS | {None}:
                    self.erro(expr.linha, f"'{expr.op}' unário não aceita {tipo}")
                expr.tipo = tipo
            return expr
        if isinstance(expr, BinOp):
            expr.esq = self.expr(expr.esq)
            expr.dir = self.expr(expr.dir)
            tipo_esq, tipo_dir = expr.esq.tipo, expr.dir.tipo
            if tipo_esq is None or tipo_dir is None:
                expr.tipo = "logico" if expr.op in self.RELACIONAIS | {"e", "ou"} else tipo_esq or tipo_dir
                return expr
            if expr.op in ("e", "ou"):
                if tipo_esq != "logico" or tipo_dir != "logico":
                    self.erro(expr.linha, f"'{expr.op}' exige dois logico, recebeu {tipo_esq} e {tipo_dir}")
                expr.tipo = "logico"
            elif expr.op in self.RELACIONAIS:
                if tipo_esq in self.NUMERICOS and tipo_dir in self.NUMERICOS:
                    if "decimal" in (tipo_esq, tipo_dir):
                        expr.esq = self.converter(expr.esq, "decimal", expr.linha, "comparação")
                        expr.dir = self.converter(expr.dir, "decimal", expr.linha, "comparação")
                elif expr.op in ("==", "!=") and tipo_esq == tipo_dir:
                    pass
                else:
                    self.erro(expr.linha, f"comparação '{expr.op}' inválida entre {tipo_esq} e {tipo_dir}")
                expr.tipo = "logico"
            elif expr.op == "+" and tipo_esq == tipo_dir == "texto":
                expr.tipo = "texto"
            elif expr.op in ("+", "-", "*", "/") and tipo_esq in self.NUMERICOS and tipo_dir in self.NUMERICOS:
                if expr.op == "/" and self.valor_constante(expr.dir) == 0:
                    self.erro(expr.linha, "divisão por zero")
                if expr.op == "/" or "decimal" in (tipo_esq, tipo_dir):
                    expr.esq = self.converter(expr.esq, "decimal", expr.linha, "operação aritmética")
                    expr.dir = self.converter(expr.dir, "decimal", expr.linha, "operação aritmética")
                    expr.tipo = "decimal"
                else:
                    expr.tipo = "inteiro"
            else:
                self.erro(expr.linha, f"operação '{expr.op}' inválida entre {tipo_esq} e {tipo_dir}")
                expr.tipo = tipo_esq
            return expr
        raise TypeError(f"Expressão AST não suportada: {type(expr).__name__}")

def analisar_semantica(fonte):
    programa = analisar_sintaxe(fonte)
    analisador = AnalisadorSemantico()
    analisador.analisar(programa)
    return programa, analisador

GALERIA_SEMANTICA = [
    ("variável não declarada", 'mostrar saldo_inexistente;'),
    ("tipo incompatível", "inteiro pontos = 3.5;"),
    ("declaração duplicada", "inteiro pontos = 10; inteiro pontos = 20;"),
    ("condição não lógica", "se 5 { mostrar 1; }"),
    ("operação entre texto e número", 'mostrar "cliente_42" - 2;'),
    ("variável fora do escopo", "se verdadeiro { inteiro bonus = 10; mostrar bonus; } mostrar bonus;"),
    ("cashback acima de 100%", "decimal percentual_cashback = 150.0; mostrar percentual_cashback;"),
    ("pontos negativos", 'creditar "cliente_42", -5;'),
]

for titulo, fonte in GALERIA_SEMANTICA:
    programa = analisar_sintaxe(fonte)
    analisador = AnalisadorSemantico()
    analisador.analisar(programa)
    if not analisador.erros:
        raise AssertionError(f"O caso '{titulo}' deveria produzir erro semântico.")
    print(f"\n--- {titulo} ---")
    for diagnostico in analisador.diagnosticos:
        print(diagnostico)

PROGRAMA_COERCAO = "decimal cashback = 10; mostrar cashback;"
ast_coercao, sem_coercao = analisar_semantica(PROGRAMA_COERCAO)
if sem_coercao.erros or not isinstance(ast_coercao.comandos[0].expr, Conv):
    raise AssertionError("A coerção inteiro para decimal não foi aplicada corretamente.")
print("\nCoerção inteiro -> decimal inserida na AST:")
mostrar_ast(ast_coercao.comandos[0])

PROGRAMA_AVISOS = 'decimal percentual_cashback = 0.0; creditar "cliente_42", 0;'
_, sem_avisos = analisar_semantica(PROGRAMA_AVISOS)
if sem_avisos.erros or not sem_avisos.avisos:
    raise AssertionError("O programa de demonstração deveria produzir avisos sem erros.")
print("\nExemplo de avisos (não bloqueiam a análise):")
for diagnostico in sem_avisos.avisos:
    print(diagnostico)



--- variável não declarada ---
[semântico] linha 1, coluna 0: 'saldo_inexistente' não foi declarada
Dica: declare a variável antes de usá-la

--- tipo incompatível ---
[semântico] linha 1, coluna 0: declaração de 'pontos': esperava inteiro, recebeu decimal
Dica: Use decimal quando o valor tiver casas decimais.
AVISO [semântico] linha 1, coluna 0: 'pontos' foi declarada e nunca usada

--- declaração duplicada ---
[semântico] linha 1, coluna 0: 'pontos' já foi declarada neste bloco
AVISO [semântico] linha 1, coluna 0: 'pontos' foi declarada e nunca usada

--- condição não lógica ---
[semântico] linha 1, coluna 0: a condição de 'se' precisa ser logico, mas é inteiro

--- operação entre texto e número ---
[semântico] linha 1, coluna 0: operação '-' inválida entre texto e inteiro

--- variável fora do escopo ---
[semântico] linha 1, coluna 0: 'bonus' não foi declarada
Dica: declare a variável antes de usá-la

--- cashback acima de 100% ---
[semântico] linha 1, coluna 0: percentual de cashb

## Passo 4 — Código de Três Endereços (TAC)

O TAC transforma a AST validada em instruções simples com temporários, rótulos e desvios. Condições de desvio são invertidas para saltar ao caminho alternativo; cada comando da FidelidadeScript mantém uma instrução própria.


In [5]:
# ============================================================
# PASSO 4 — Geração de Código de Três Endereços (TAC)
# ============================================================

@dataclass(frozen=True)
class VariavelTAC:
    nome: str

@dataclass
class InstrucaoTAC:
    op: str
    arg1: object = None
    arg2: object = None
    destino: object = None
    arg3: object = None

class GeradorTAC:
    INVERSO = {">": "<=", "<": ">=", ">=": "<", "<=": ">", "==": "!=", "!=": "=="}
    CONDICIONAIS = {">": "GT", "<": "LT", ">=": "GE", "<=": "LE", "==": "EQ", "!=": "NE"}
    OPERADORES = {
        "+": "ADD", "-": "SUB", "*": "MUL", "/": "DIV",
        ">": "GT", "<": "LT", ">=": "GE", "<=": "LE",
        "==": "EQ", "!=": "NE", "e": "AND", "ou": "OR",
    }

    def __init__(self):
        self.codigo = []
        self.escopos = []
        self.proximo_temporario = 0
        self.proximo_rotulo = 0
        self.proxima_variavel = 0

    def novo_temporario(self):
        self.proximo_temporario += 1
        return VariavelTAC(f"$t{self.proximo_temporario}")

    def novo_rotulo(self):
        self.proximo_rotulo += 1
        return f"L{self.proximo_rotulo}"

    def emitir(self, op, arg1=None, arg2=None, destino=None, arg3=None):
        self.codigo.append(InstrucaoTAC(op, arg1, arg2, destino, arg3))

    def resolver(self, nome):
        for escopo in reversed(self.escopos):
            if nome in escopo:
                return VariavelTAC(escopo[nome])
        raise ValueError(f"Variável sem símbolo no TAC: {nome}")

    def declarar(self, nome):
        atual = self.escopos[-1]
        if nome in atual:
            raise ValueError(f"Variável já declarada no escopo TAC: {nome}")
        if len(self.escopos) == 1:
            nome_tac = nome
        else:
            self.proxima_variavel += 1
            nome_tac = f"$v{self.proxima_variavel}.{nome}"
        atual[nome] = nome_tac
        return VariavelTAC(nome_tac)

    def expressao(self, expr):
        if isinstance(expr, Lit):
            return expr.valor
        if isinstance(expr, Var):
            return self.resolver(expr.nome)
        if isinstance(expr, Conv):
            origem = self.expressao(expr.expr)
            resultado = self.novo_temporario()
            self.emitir("I2F", origem, destino=resultado)
            return resultado
        if isinstance(expr, UnOp):
            origem = self.expressao(expr.expr)
            if expr.op == "+":
                return origem
            operador = {"-": "NEG", "nao": "NOT"}.get(expr.op)
            if operador is None:
                raise ValueError(f"Operador unário TAC não suportado: {expr.op}")
            resultado = self.novo_temporario()
            self.emitir(operador, origem, destino=resultado)
            return resultado
        if isinstance(expr, BinOp):
            esquerda = self.expressao(expr.esq)
            direita = self.expressao(expr.dir)
            operador = self.OPERADORES.get(expr.op)
            if operador is None:
                raise ValueError(f"Operador TAC não suportado: {expr.op}")
            resultado = self.novo_temporario()
            self.emitir(operador, esquerda, direita, resultado)
            return resultado
        raise TypeError(f"Expressão AST não suportada pelo TAC: {type(expr).__name__}")

    def condicao(self, expr, verdadeiro, falso):
        if isinstance(expr, UnOp) and expr.op == "nao":
            self.condicao(expr.expr, falso, verdadeiro)
            return
        if isinstance(expr, BinOp) and expr.op == "e":
            proxima = self.novo_rotulo()
            self.condicao(expr.esq, proxima, falso)
            self.emitir("LABEL", destino=proxima)
            self.condicao(expr.dir, verdadeiro, falso)
            return
        if isinstance(expr, BinOp) and expr.op == "ou":
            proxima = self.novo_rotulo()
            self.condicao(expr.esq, verdadeiro, proxima)
            self.emitir("LABEL", destino=proxima)
            self.condicao(expr.dir, verdadeiro, falso)
            return
        if isinstance(expr, BinOp) and expr.op in self.INVERSO:
            esquerda = self.expressao(expr.esq)
            direita = self.expressao(expr.dir)
            operador_invertido = self.INVERSO[expr.op]
            instrucao = self.CONDICIONAIS[operador_invertido]
            self.emitir(f"IF_{instrucao}", esquerda, direita, falso)
            self.emitir("GOTO", destino=verdadeiro)
            return
        valor = self.expressao(expr)
        self.emitir("IF_FALSE", valor, destino=falso)
        self.emitir("GOTO", destino=verdadeiro)

    def bloco(self, comandos, novo_escopo=True):
        if novo_escopo:
            self.escopos.append({})
        try:
            for comando in comandos:
                self.comando(comando)
        finally:
            if novo_escopo:
                self.escopos.pop()

    def comando(self, comando):
        if isinstance(comando, Decl):
            destino = self.declarar(comando.nome)
            self.emitir("DECL", comando.tipo, destino=destino)
            if comando.expr is not None:
                self.emitir("MOV", self.expressao(comando.expr), destino=destino)
        elif isinstance(comando, Atrib):
            self.emitir("MOV", self.expressao(comando.expr), destino=self.resolver(comando.nome))
        elif isinstance(comando, Mostrar):
            self.emitir("PRINT", self.expressao(comando.expr))
        elif isinstance(comando, Se):
            rotulo_entao = self.novo_rotulo()
            rotulo_fim = self.novo_rotulo()
            rotulo_senao = self.novo_rotulo() if comando.senao is not None else rotulo_fim
            self.condicao(comando.cond, rotulo_entao, rotulo_senao)
            self.emitir("LABEL", destino=rotulo_entao)
            self.bloco(comando.entao)
            if comando.senao is not None:
                self.emitir("GOTO", destino=rotulo_fim)
                self.emitir("LABEL", destino=rotulo_senao)
                self.bloco(comando.senao)
            self.emitir("LABEL", destino=rotulo_fim)
        elif isinstance(comando, Enquanto):
            rotulo_inicio = self.novo_rotulo()
            rotulo_corpo = self.novo_rotulo()
            rotulo_fim = self.novo_rotulo()
            self.emitir("LABEL", destino=rotulo_inicio)
            self.condicao(comando.cond, rotulo_corpo, rotulo_fim)
            self.emitir("LABEL", destino=rotulo_corpo)
            self.bloco(comando.corpo)
            self.emitir("GOTO", destino=rotulo_inicio)
            self.emitir("LABEL", destino=rotulo_fim)
        elif isinstance(comando, Creditar):
            cliente = self.expressao(comando.cliente)
            pontos = self.expressao(comando.pontos)
            self.emitir("CREDITAR", cliente, pontos)
        elif isinstance(comando, Resgatar):
            cliente = self.expressao(comando.cliente)
            cupom = self.expressao(comando.cupom)
            pontos = self.expressao(comando.pontos)
            self.emitir("RESGATAR", cliente, cupom, arg3=pontos)
        elif isinstance(comando, Expirar):
            cliente = self.expressao(comando.cliente)
            pontos = self.expressao(comando.pontos)
            self.emitir("EXPIRAR", cliente, pontos)
        elif isinstance(comando, Extrato):
            self.emitir("EXTRATO", self.expressao(comando.cliente))
        else:
            raise TypeError(f"Comando AST não suportado pelo TAC: {type(comando).__name__}")

    def gerar(self, programa):
        if not isinstance(programa, Programa):
            raise TypeError("gerar_tac espera um nó Programa")
        self.codigo = []
        self.escopos = [{}]
        self.proximo_temporario = 0
        self.proximo_rotulo = 0
        self.proxima_variavel = 0
        self.bloco(programa.comandos, novo_escopo=False)
        return list(self.codigo)


def gerar_tac(programa):
    return GeradorTAC().gerar(programa)


def formatar_operando_tac(operando):
    if isinstance(operando, VariavelTAC):
        return operando.nome
    if isinstance(operando, bool):
        return "verdadeiro" if operando else "falso"
    if isinstance(operando, str):
        return repr(operando)
    if operando is None:
        return "-"
    return str(operando)


def formatar_instrucao_tac(instrucao):
    op = instrucao.op
    arg1 = formatar_operando_tac(instrucao.arg1)
    arg2 = formatar_operando_tac(instrucao.arg2)
    arg3 = formatar_operando_tac(instrucao.arg3)
    destino = formatar_operando_tac(instrucao.destino)
    if op == "LABEL":
        return f"{instrucao.destino}:"
    if op == "GOTO":
        return f"GOTO {instrucao.destino}"
    if op == "IF_FALSE":
        return f"IF_FALSE {arg1} GOTO {instrucao.destino}"
    if op.startswith("IF_"):
        nome_operador = op[3:]
        simbolos = {valor: chave for chave, valor in GeradorTAC.CONDICIONAIS.items()}
        if nome_operador not in simbolos:
            raise ValueError(f"Desvio TAC não reconhecido: {op}")
        return f"IF {arg1} {simbolos[nome_operador]} {arg2} GOTO {instrucao.destino}"
    if op == "DECL":
        return f"DECL {instrucao.arg1} {destino}"
    if op == "MOV":
        return f"{destino} = {arg1}"
    if op == "PRINT":
        return f"MOSTRAR {arg1}"
    if op == "I2F":
        return f"{destino} = I2F({arg1})"
    if op == "NEG":
        return f"{destino} = -{arg1}"
    if op == "NOT":
        return f"{destino} = nao {arg1}"
    operadores = {valor: chave for chave, valor in GeradorTAC.OPERADORES.items()}
    if op in operadores:
        return f"{destino} = {arg1} {operadores[op]} {arg2}"
    if op in {"CREDITAR", "EXPIRAR"}:
        return f"{op} {arg1}, {arg2}"
    if op == "RESGATAR":
        return f"RESGATAR {arg1}, {arg2}, {arg3}"
    if op == "EXTRATO":
        return f"EXTRATO {arg1}"
    raise ValueError(f"Instrução TAC não reconhecida: {op}")


def mostrar_tac(programa):
    codigo = gerar_tac(programa)
    print("Código de Três Endereços (TAC):")
    for instrucao in codigo:
        print("  " + formatar_instrucao_tac(instrucao))
    return codigo


PROGRAMA_TAC = r'''inteiro pontos = 3;
decimal compra = 250.0;
se compra >= 200.0 e pontos > 0 {
    creditar "cliente_42", pontos;
} senao {
    extrato "cliente_42";
}
enquanto pontos > 0 {
    resgatar "cliente_42", "cupom_10", 1;
    expirar "cliente_42", 1;
    pontos = pontos - 1;
}
extrato "cliente_42";
'''

AST_TAC, SEMANTICA_TAC = analisar_semantica(PROGRAMA_TAC)
if SEMANTICA_TAC.erros:
    raise AssertionError("O programa de demonstração TAC deve passar pela análise semântica.")
CODIGO_TAC = mostrar_tac(AST_TAC)
OPERACOES_TAC = {instrucao.op for instrucao in CODIGO_TAC}
if not {"CREDITAR", "RESGATAR", "EXPIRAR", "EXTRATO"}.issubset(OPERACOES_TAC):
    raise AssertionError("Cada comando do domínio deve possuir sua própria instrução TAC.")
if not {"IF_LT", "IF_LE"}.issubset(OPERACOES_TAC) or "GOTO" not in OPERACOES_TAC or "LABEL" not in OPERACOES_TAC:
    raise AssertionError("A demonstração deve exibir desvios invertidos, rótulos e saltos.")

AST_CONVERSAO_TAC, SEMANTICA_CONVERSAO_TAC = analisar_semantica("decimal valor = 2; mostrar valor;")
if SEMANTICA_CONVERSAO_TAC.erros or "I2F" not in {i.op for i in gerar_tac(AST_CONVERSAO_TAC)}:
    raise AssertionError("A coerção inteiro para decimal deve ser representada no TAC.")


Código de Três Endereços (TAC):
  DECL inteiro pontos
  pontos = 3
  DECL decimal compra
  compra = 250.0
  IF compra < 200.0 GOTO L3
  GOTO L4
  L4:
  IF pontos <= 0 GOTO L3
  GOTO L1
  L1:
  CREDITAR 'cliente_42', pontos
  GOTO L2
  L3:
  EXTRATO 'cliente_42'
  L2:
  L5:
  IF pontos <= 0 GOTO L7
  GOTO L6
  L6:
  RESGATAR 'cliente_42', 'cupom_10', 1
  EXPIRAR 'cliente_42', 1
  $t1 = pontos - 1
  pontos = $t1
  GOTO L5
  L7:
  EXTRATO 'cliente_42'


## Passo 5 — Otimização do TAC

O otimizador aplica folding e propagação de constantes, simplificação algébrica e eliminação de temporários mortos. As demonstrações executam o TAC original e o otimizado, comparam a saída e contam as instruções executadas.


In [6]:
# ============================================================
# PASSO 5 — Otimização do TAC (R6)
# ============================================================

_OPERADORES_FOLD_TAC = {"ADD", "SUB", "MUL", "GT", "LT", "GE", "LE", "EQ", "NE", "AND", "OR"}
_OPERADORES_BINARIOS_TAC = _OPERADORES_FOLD_TAC | {"DIV"}
_OPERADORES_UNARIOS_TAC = {"NEG", "NOT"}
_MANTER_TAC = object()
_SEM_RESULTADO_TAC = object()


def eh_imediato_tac(valor):
    return valor is not None and not isinstance(valor, VariavelTAC)


def copiar_instrucao_tac(instrucao, arg1=_MANTER_TAC, arg2=_MANTER_TAC, arg3=_MANTER_TAC):
    return InstrucaoTAC(
        instrucao.op,
        instrucao.arg1 if arg1 is _MANTER_TAC else arg1,
        instrucao.arg2 if arg2 is _MANTER_TAC else arg2,
        instrucao.destino,
        instrucao.arg3 if arg3 is _MANTER_TAC else arg3,
    )


def calcular_binario_tac(op, esquerda, direita):
    if op == "ADD": return esquerda + direita
    if op == "SUB": return esquerda - direita
    if op == "MUL": return esquerda * direita
    if op == "GT": return esquerda > direita
    if op == "LT": return esquerda < direita
    if op == "GE": return esquerda >= direita
    if op == "LE": return esquerda <= direita
    if op == "EQ": return esquerda == direita
    if op == "NE": return esquerda != direita
    if op == "AND": return esquerda and direita
    if op == "OR": return esquerda or direita
    return _SEM_RESULTADO_TAC


def dobrar_constantes_tac(codigo):
    resultado = []
    for instrucao in codigo:
        if instrucao.op in _OPERADORES_BINARIOS_TAC and instrucao.op != "DIV":
            if eh_imediato_tac(instrucao.arg1) and eh_imediato_tac(instrucao.arg2):
                valor = calcular_binario_tac(instrucao.op, instrucao.arg1, instrucao.arg2)
                if valor is not _SEM_RESULTADO_TAC:
                    resultado.append(InstrucaoTAC("MOV", valor, destino=instrucao.destino))
                    continue
        elif instrucao.op == "NEG" and eh_imediato_tac(instrucao.arg1):
            resultado.append(InstrucaoTAC("MOV", -instrucao.arg1, destino=instrucao.destino))
            continue
        elif instrucao.op == "NOT" and isinstance(instrucao.arg1, bool):
            resultado.append(InstrucaoTAC("MOV", not instrucao.arg1, destino=instrucao.destino))
            continue
        resultado.append(instrucao)
    return resultado


def propagar_constantes_tac(codigo):
    constantes = {}
    resultado = []
    for instrucao in codigo:
        if instrucao.op == "LABEL":
            constantes.clear()
            resultado.append(instrucao)
            continue
        arg1 = constantes.get(instrucao.arg1, instrucao.arg1) if isinstance(instrucao.arg1, VariavelTAC) else instrucao.arg1
        arg2 = constantes.get(instrucao.arg2, instrucao.arg2) if isinstance(instrucao.arg2, VariavelTAC) else instrucao.arg2
        arg3 = constantes.get(instrucao.arg3, instrucao.arg3) if isinstance(instrucao.arg3, VariavelTAC) else instrucao.arg3
        resultado.append(copiar_instrucao_tac(instrucao, arg1, arg2, arg3))
        destino = instrucao.destino
        if instrucao.op == "DECL" and isinstance(destino, VariavelTAC):
            constantes.pop(destino, None)
        elif instrucao.op == "MOV" and isinstance(destino, VariavelTAC):
            constantes.pop(destino, None)
            if eh_imediato_tac(arg1): constantes[destino] = arg1
        elif instrucao.op in _OPERADORES_UNARIOS_TAC and isinstance(destino, VariavelTAC):
            constantes.pop(destino, None)
            if instrucao.op == "NEG" and eh_imediato_tac(arg1): constantes[destino] = -arg1
            elif instrucao.op == "NOT" and isinstance(arg1, bool): constantes[destino] = not arg1
        elif instrucao.op in _OPERADORES_BINARIOS_TAC and isinstance(destino, VariavelTAC):
            constantes.pop(destino, None)
            if instrucao.op != "DIV" and eh_imediato_tac(arg1) and eh_imediato_tac(arg2):
                valor = calcular_binario_tac(instrucao.op, arg1, arg2)
                if valor is not _SEM_RESULTADO_TAC: constantes[destino] = valor
        elif instrucao.op.startswith("IF_") or instrucao.op == "GOTO":
            constantes.clear()
    return resultado


def numero_igual_tac(valor, esperado):
    return isinstance(valor, (int, float)) and not isinstance(valor, bool) and valor == esperado


def simplificar_algebra_tac(codigo):
    resultado = []
    for instrucao in codigo:
        op, esquerda, direita = instrucao.op, instrucao.arg1, instrucao.arg2
        substituto = _MANTER_TAC
        if op == "ADD":
            if numero_igual_tac(esquerda, 0): substituto = direita
            elif numero_igual_tac(direita, 0): substituto = esquerda
        elif op == "SUB" and numero_igual_tac(direita, 0):
            substituto = esquerda
        elif op == "MUL":
            if numero_igual_tac(esquerda, 1): substituto = direita
            elif numero_igual_tac(direita, 1): substituto = esquerda
        if substituto is not _MANTER_TAC:
            resultado.append(InstrucaoTAC("MOV", substituto, destino=instrucao.destino))
        else:
            resultado.append(instrucao)
    return resultado


def eh_temporario_tac(variavel):
    return isinstance(variavel, VariavelTAC) and variavel.nome.startswith("$t")


def eliminar_temporarios_mortos_tac(codigo):
    vivos = set()
    resultado_reverso = []
    definicoes_puras = _OPERADORES_FOLD_TAC | _OPERADORES_UNARIOS_TAC | {"MOV"}
    for instrucao in reversed(codigo):
        if instrucao.op == "MOV" and isinstance(instrucao.arg1, VariavelTAC) and instrucao.arg1 == instrucao.destino:
            continue
        destino = instrucao.destino
        if eh_temporario_tac(destino) and instrucao.op in definicoes_puras and destino not in vivos:
            continue
        if isinstance(destino, VariavelTAC): vivos.discard(destino)
        for operando in (instrucao.arg1, instrucao.arg2, instrucao.arg3):
            if isinstance(operando, VariavelTAC): vivos.add(operando)
        resultado_reverso.append(instrucao)
    return list(reversed(resultado_reverso))


def otimizar_tac(codigo):
    atual = list(codigo)
    for _ in range(8):
        otimizado = dobrar_constantes_tac(atual)
        otimizado = propagar_constantes_tac(otimizado)
        otimizado = simplificar_algebra_tac(otimizado)
        otimizado = propagar_constantes_tac(otimizado)
        otimizado = eliminar_temporarios_mortos_tac(otimizado)
        if otimizado == atual: return otimizado
        atual = otimizado
    return atual


def executar_tac_otimizacao(codigo):
    memoria = {}
    saida = []
    executadas = 0
    for instrucao in codigo:
        executadas += 1
        if instrucao.op == "DECL":
            memoria[instrucao.destino] = None
        elif instrucao.op == "MOV":
            valor = memoria[instrucao.arg1] if isinstance(instrucao.arg1, VariavelTAC) else instrucao.arg1
            memoria[instrucao.destino] = valor
        elif instrucao.op == "PRINT":
            valor = memoria[instrucao.arg1] if isinstance(instrucao.arg1, VariavelTAC) else instrucao.arg1
            saida.append(valor)
        elif instrucao.op in _OPERADORES_BINARIOS_TAC:
            esquerda = memoria[instrucao.arg1] if isinstance(instrucao.arg1, VariavelTAC) else instrucao.arg1
            direita = memoria[instrucao.arg2] if isinstance(instrucao.arg2, VariavelTAC) else instrucao.arg2
            if instrucao.op == "DIV":
                if direita == 0: raise ZeroDivisionError("divisão por zero durante a validação do TAC")
                valor = esquerda / direita
            else:
                valor = calcular_binario_tac(instrucao.op, esquerda, direita)
            memoria[instrucao.destino] = valor
        elif instrucao.op in _OPERADORES_UNARIOS_TAC:
            origem = memoria[instrucao.arg1] if isinstance(instrucao.arg1, VariavelTAC) else instrucao.arg1
            memoria[instrucao.destino] = -origem if instrucao.op == "NEG" else not origem
        elif instrucao.op == "I2F":
            origem = memoria[instrucao.arg1] if isinstance(instrucao.arg1, VariavelTAC) else instrucao.arg1
            memoria[instrucao.destino] = float(origem)
        else:
            raise ValueError(f"O validador R6 não suporta a instrução {instrucao.op}")
    return {"saida": saida, "instrucoes_executadas": executadas}


def saidas_iguais_tac(saida_a, saida_b):
    return len(saida_a) == len(saida_b) and all(
        type(a) is type(b) and a == b for a, b in zip(saida_a, saida_b)
    )


def exibir_codigo_tac(titulo, codigo):
    print(titulo)
    for instrucao in codigo:
        print("  " + formatar_instrucao_tac(instrucao))


PROGRAMAS_OTIMIZACAO = [
    ("folding e propagação de constantes", "inteiro total = 2 + 3; mostrar total;"),
    ("simplificação algébrica", "inteiro pontos = 10; pontos = pontos + 0; mostrar pontos;"),
]

RESULTADOS_OTIMIZACAO = []
for titulo, fonte in PROGRAMAS_OTIMIZACAO:
    ast, semantica = analisar_semantica(fonte)
    if semantica.erros:
        raise AssertionError(f"O programa de otimização '{titulo}' deve ser semanticamente válido.")
    codigo_original = gerar_tac(ast)
    codigo_otimizado = otimizar_tac(codigo_original)
    resultado_original = executar_tac_otimizacao(codigo_original)
    resultado_otimizado = executar_tac_otimizacao(codigo_otimizado)
    if not saidas_iguais_tac(resultado_original["saida"], resultado_otimizado["saida"]):
        raise AssertionError(f"A otimização alterou a saída de '{titulo}'.")
    if resultado_otimizado["instrucoes_executadas"] >= resultado_original["instrucoes_executadas"]:
        raise AssertionError(f"A otimização não reduziu instruções em '{titulo}'.")
    exibir_codigo_tac(titulo + " — antes:", codigo_original)
    exibir_codigo_tac("Depois:", codigo_otimizado)
    print(f"Saída: {resultado_original['saida']!r} | instruções executadas: "
          f"{resultado_original['instrucoes_executadas']} -> {resultado_otimizado['instrucoes_executadas']}")
    RESULTADOS_OTIMIZACAO.append({
        "nome": titulo,
        "saida": resultado_original["saida"],
        "instrucoes_antes": resultado_original["instrucoes_executadas"],
        "instrucoes_depois": resultado_otimizado["instrucoes_executadas"],
        "tac_antes": codigo_original,
        "tac_depois": codigo_otimizado,
    })

if len(RESULTADOS_OTIMIZACAO) < 2:
    raise AssertionError("R6 exige pelo menos dois programas de comparação.")
print("Técnicas: folding de constantes, propagação de constantes, "
      "simplificação algébrica e eliminação de temporários mortos.")


folding e propagação de constantes — antes:
  DECL inteiro total
  $t1 = 2 + 3
  total = $t1
  MOSTRAR total
Depois:
  DECL inteiro total
  total = 5
  MOSTRAR 5
Saída: [5] | instruções executadas: 4 -> 3
simplificação algébrica — antes:
  DECL inteiro pontos
  pontos = 10
  $t1 = pontos + 0
  pontos = $t1
  MOSTRAR pontos
Depois:
  DECL inteiro pontos
  pontos = 10
  pontos = 10
  MOSTRAR 10
Saída: [10] | instruções executadas: 5 -> 4
Técnicas: folding de constantes, propagação de constantes, simplificação algébrica e eliminação de temporários mortos.


## Passo 6 — Execução: máquina virtual do TAC

A máquina virtual executa instruções TAC, mantém saldos por cliente, produz saída para os comandos de domínio e agrega estatísticas. `resgatar` recebe cliente, cupom e pontos; saldo insuficiente e limites de execução geram erros tratados.


In [7]:
# ============================================================
# PASSO 6 — Execução do TAC com a máquina virtual (R7)
# ============================================================

@dataclass
class ResultadoExecucao:
    saida: list
    estatisticas: dict
    memoria: dict
    saldos: dict


class ErroExecucao(RuntimeError):
    def __init__(self, mensagem, pc=None, operacao=None):
        self.mensagem = mensagem
        self.pc = pc
        self.operacao = operacao
        contexto = "" if pc is None else f" (PC {pc}, instrução {operacao})"
        super().__init__(f"[execução] {mensagem}{contexto}")


class MaquinaVirtual:
    TIPOS_PADRAO = {"inteiro": 0, "decimal": 0.0, "texto": "", "logico": False}
    OPERADORES_DESVIO = {"IF_GT": "GT", "IF_LT": "LT", "IF_GE": "GE", "IF_LE": "LE", "IF_EQ": "EQ", "IF_NE": "NE"}

    def __init__(self, limite_instrucoes=10000):
        if not isinstance(limite_instrucoes, int) or isinstance(limite_instrucoes, bool) or limite_instrucoes < 1:
            raise ValueError("limite_instrucoes deve ser um inteiro positivo")
        self.limite_instrucoes = limite_instrucoes

    def _falhar(self, mensagem, pc, instrucao):
        raise ErroExecucao(mensagem, pc, instrucao.op)

    def _ler(self, operando, memoria, pc, instrucao):
        if isinstance(operando, VariavelTAC):
            if operando not in memoria:
                self._falhar(f"variável ou temporário não inicializado: {operando.nome}", pc, instrucao)
            return memoria[operando]
        if isinstance(operando, (str, int, float, bool)):
            return operando
        self._falhar("operando TAC inválido", pc, instrucao)

    def _gravar(self, destino, valor, memoria, pc, instrucao):
        if not isinstance(destino, VariavelTAC):
            self._falhar("destino TAC inválido", pc, instrucao)
        memoria[destino] = valor

    def _cliente(self, valor, pc, instrucao):
        if not isinstance(valor, str) or not valor.strip():
            self._falhar("o identificador do cliente deve ser texto não vazio", pc, instrucao)
        return valor

    def _pontos(self, valor, pc, instrucao):
        if not isinstance(valor, int) or isinstance(valor, bool):
            self._falhar("a quantidade de pontos deve ser inteira", pc, instrucao)
        if valor < 0:
            self._falhar("pontos negativos não são permitidos", pc, instrucao)
        return valor

    @staticmethod
    def _eh_percentual_cashback(destino):
        return isinstance(destino, VariavelTAC) and (
            destino.nome == "percentual_cashback"
            or destino.nome.endswith(".percentual_cashback")
        )

    def _validar_percentual_cashback(self, destino, valor, pc, instrucao):
        if (
            self._eh_percentual_cashback(destino)
            and isinstance(valor, (int, float))
            and not isinstance(valor, bool)
            and valor > 100
        ):
            self._falhar(
                f"percentual de cashback inválido ({valor}%): o limite é 100%",
                pc,
                instrucao,
            )

    @staticmethod
    def _formatar_valor(valor):
        if isinstance(valor, bool):
            return "verdadeiro" if valor else "falso"
        return str(valor)

    def _avaliar_binario(self, op, esquerda, direita, pc, instrucao):
        if op == "DIV" and direita == 0:
            self._falhar("divisão por zero", pc, instrucao)
        try:
            if op == "DIV":
                return esquerda / direita
            valor = calcular_binario_tac(op, esquerda, direita)
        except (TypeError, ValueError, OverflowError) as erro:
            self._falhar(f"operação {op} inválida: {erro}", pc, instrucao)
        if valor is _SEM_RESULTADO_TAC:
            self._falhar(f"operação TAC não suportada: {op}", pc, instrucao)
        return valor

    def executar(self, codigo):
        instrucoes = list(codigo)
        rotulos = {}
        for indice, instrucao in enumerate(instrucoes):
            if not isinstance(instrucao, InstrucaoTAC):
                raise TypeError("A máquina virtual espera uma sequência de InstrucaoTAC")
            if instrucao.op == "LABEL":
                if not isinstance(instrucao.destino, str) or instrucao.destino in rotulos:
                    raise ErroExecucao("rótulo ausente ou duplicado", indice, instrucao.op)
                rotulos[instrucao.destino] = indice
        for indice, instrucao in enumerate(instrucoes):
            if instrucao.op == "GOTO" or instrucao.op == "IF_FALSE" or instrucao.op in self.OPERADORES_DESVIO:
                if instrucao.destino not in rotulos:
                    raise ErroExecucao("desvio aponta para rótulo inexistente", indice, instrucao.op)

        memoria = {}
        saldos = {}
        saida = []
        estatisticas = {
            "instrucoes_executadas": 0,
            "comandos_creditar": 0,
            "pontos_creditados": 0,
            "comandos_resgatar": 0,
            "pontos_resgatados": 0,
            "comandos_expirar": 0,
            "pontos_expirados": 0,
            "comandos_extrato": 0,
        }
        pc = 0
        while pc < len(instrucoes):
            if estatisticas["instrucoes_executadas"] >= self.limite_instrucoes:
                instrucao = instrucoes[pc]
                self._falhar(
                    f"limite de {self.limite_instrucoes} instruções atingido; possível laço infinito",
                    pc,
                    instrucao,
                )
            estatisticas["instrucoes_executadas"] += 1
            instrucao = instrucoes[pc]
            op = instrucao.op
            if op == "LABEL":
                pc += 1
            elif op == "GOTO":
                pc = rotulos[instrucao.destino]
            elif op == "IF_FALSE":
                condicao = self._ler(instrucao.arg1, memoria, pc, instrucao)
                if not isinstance(condicao, bool):
                    self._falhar("IF_FALSE exige uma condição lógica", pc, instrucao)
                pc = rotulos[instrucao.destino] if not condicao else pc + 1
            elif op in self.OPERADORES_DESVIO:
                esquerda = self._ler(instrucao.arg1, memoria, pc, instrucao)
                direita = self._ler(instrucao.arg2, memoria, pc, instrucao)
                teste = self._avaliar_binario(self.OPERADORES_DESVIO[op], esquerda, direita, pc, instrucao)
                pc = rotulos[instrucao.destino] if teste else pc + 1
            elif op == "DECL":
                if instrucao.arg1 not in self.TIPOS_PADRAO:
                    self._falhar(f"tipo de declaração desconhecido: {instrucao.arg1}", pc, instrucao)
                self._gravar(instrucao.destino, self.TIPOS_PADRAO[instrucao.arg1], memoria, pc, instrucao)
                pc += 1
            elif op == "MOV":
                valor = self._ler(instrucao.arg1, memoria, pc, instrucao)
                self._validar_percentual_cashback(instrucao.destino, valor, pc, instrucao)
                self._gravar(instrucao.destino, valor, memoria, pc, instrucao)
                pc += 1
            elif op == "PRINT":
                valor = self._ler(instrucao.arg1, memoria, pc, instrucao)
                saida.append(self._formatar_valor(valor))
                pc += 1
            elif op in _OPERADORES_BINARIOS_TAC:
                esquerda = self._ler(instrucao.arg1, memoria, pc, instrucao)
                direita = self._ler(instrucao.arg2, memoria, pc, instrucao)
                valor = self._avaliar_binario(op, esquerda, direita, pc, instrucao)
                self._gravar(instrucao.destino, valor, memoria, pc, instrucao)
                pc += 1
            elif op in _OPERADORES_UNARIOS_TAC:
                origem = self._ler(instrucao.arg1, memoria, pc, instrucao)
                if op == "NEG":
                    if not isinstance(origem, (int, float)) or isinstance(origem, bool):
                        self._falhar("NEG exige um valor numérico", pc, instrucao)
                    valor = -origem
                else:
                    if not isinstance(origem, bool):
                        self._falhar("NOT exige um valor lógico", pc, instrucao)
                    valor = not origem
                self._gravar(instrucao.destino, valor, memoria, pc, instrucao)
                pc += 1
            elif op == "I2F":
                origem = self._ler(instrucao.arg1, memoria, pc, instrucao)
                if not isinstance(origem, int) or isinstance(origem, bool):
                    self._falhar("I2F exige um valor inteiro", pc, instrucao)
                try:
                    valor = float(origem)
                except OverflowError:
                    self._falhar("valor inteiro fora do intervalo decimal", pc, instrucao)
                self._gravar(instrucao.destino, valor, memoria, pc, instrucao)
                pc += 1
            elif op == "CREDITAR":
                cliente = self._cliente(self._ler(instrucao.arg1, memoria, pc, instrucao), pc, instrucao)
                pontos = self._pontos(self._ler(instrucao.arg2, memoria, pc, instrucao), pc, instrucao)
                saldos[cliente] = saldos.get(cliente, 0) + pontos
                estatisticas["comandos_creditar"] += 1
                estatisticas["pontos_creditados"] += pontos
                saida.append(f"Crédito: {cliente} recebeu {pontos} pontos (saldo: {saldos[cliente]}).")
                pc += 1
            elif op == "RESGATAR":
                cliente = self._cliente(self._ler(instrucao.arg1, memoria, pc, instrucao), pc, instrucao)
                cupom = self._ler(instrucao.arg2, memoria, pc, instrucao)
                pontos = self._pontos(self._ler(instrucao.arg3, memoria, pc, instrucao), pc, instrucao)
                if not isinstance(cupom, str) or not cupom.strip():
                    self._falhar("o identificador do cupom deve ser texto não vazio", pc, instrucao)
                saldo = saldos.get(cliente, 0)
                if pontos > saldo:
                    self._falhar(f"saldo insuficiente para resgatar {pontos} pontos de {cliente} (saldo: {saldo})", pc, instrucao)
                saldos[cliente] = saldo - pontos
                estatisticas["comandos_resgatar"] += 1
                estatisticas["pontos_resgatados"] += pontos
                saida.append(f"Resgate: cupom {cupom} usado por {cliente}; {pontos} pontos (saldo: {saldos[cliente]}).")
                pc += 1
            elif op == "EXPIRAR":
                cliente = self._cliente(self._ler(instrucao.arg1, memoria, pc, instrucao), pc, instrucao)
                pontos = self._pontos(self._ler(instrucao.arg2, memoria, pc, instrucao), pc, instrucao)
                saldo = saldos.get(cliente, 0)
                if pontos > saldo:
                    self._falhar(f"saldo insuficiente para expirar {pontos} pontos de {cliente} (saldo: {saldo})", pc, instrucao)
                saldos[cliente] = saldo - pontos
                estatisticas["comandos_expirar"] += 1
                estatisticas["pontos_expirados"] += pontos
                saida.append(f"Expiração: {pontos} pontos expiraram para {cliente} (saldo: {saldos[cliente]}).")
                pc += 1
            elif op == "EXTRATO":
                cliente = self._cliente(self._ler(instrucao.arg1, memoria, pc, instrucao), pc, instrucao)
                saldo = saldos.get(cliente, 0)
                estatisticas["comandos_extrato"] += 1
                saida.append(f"Extrato: {cliente} possui {saldo} pontos.")
                pc += 1
            else:
                self._falhar(f"instrução TAC desconhecida: {op}", pc, instrucao)

        estatisticas["clientes_com_saldo"] = len(saldos)
        estatisticas["saldo_total"] = sum(saldos.values())
        return ResultadoExecucao(saida, estatisticas, memoria.copy(), saldos.copy())


def mostrar_resultado_execucao(resultado):
    print("Saída da máquina virtual:")
    for linha in resultado.saida:
        print("  " + linha)
    print("Resumo:")
    for nome, valor in resultado.estatisticas.items():
        print(f"  {nome}: {valor}")


PROGRAMA_EXECUCAO = r'''inteiro pontos = 2;
creditar "cliente_42", pontos;
se pontos > 1 {
    resgatar "cliente_42", "cupom_10", 1;
} senao {
    mostrar 0;
}
pontos = 1;
enquanto pontos > 0 {
    expirar "cliente_42", 1;
    pontos = pontos - 1;
}
extrato "cliente_42";
mostrar pontos;
'''

AST_EXECUCAO, SEMANTICA_EXECUCAO = analisar_semantica(PROGRAMA_EXECUCAO)
if SEMANTICA_EXECUCAO.erros:
    raise AssertionError("O programa de demonstração deve passar pela análise semântica.")
RESULTADO_EXECUCAO = MaquinaVirtual().executar(gerar_tac(AST_EXECUCAO))
mostrar_resultado_execucao(RESULTADO_EXECUCAO)
ESTATISTICAS_EXECUCAO = RESULTADO_EXECUCAO.estatisticas
if not all(ESTATISTICAS_EXECUCAO[chave] == esperado for chave, esperado in {
    "comandos_creditar": 1, "comandos_resgatar": 1,
    "comandos_expirar": 1, "comandos_extrato": 1,
}.items()):
    raise AssertionError("Cada comando de domínio deve atualizar sua estatística.")
if RESULTADO_EXECUCAO.saldos.get("cliente_42") != 0:
    raise AssertionError("O saldo final do cliente está incorreto.")

# Confere que a VM executa também os TAC otimizados produzidos no R6.
for comparacao in RESULTADOS_OTIMIZACAO:
    antes = MaquinaVirtual().executar(comparacao["tac_antes"])
    depois = MaquinaVirtual().executar(comparacao["tac_depois"])
    if antes.saida != depois.saida or depois.estatisticas["instrucoes_executadas"] >= antes.estatisticas["instrucoes_executadas"]:
        raise AssertionError("A VM e o otimizador R6 divergem na saída ou nas contagens.")

ERROS_EXECUCAO_DEMONSTRACAO = [
    ("divisão por zero", "decimal saldo = 1.0; se saldo > 0.0 { saldo = 0.0; } senao { saldo = 1.0; } decimal calculado = 1.0 / saldo; mostrar calculado;", 100, "divisão por zero"),
    ("saldo insuficiente no resgate", 'creditar "cliente_42", 2; resgatar "cliente_42", "cupom_10", 3;', 100, "saldo insuficiente para resgatar"),
    ("laço infinito", "enquanto verdadeiro { }", 30, "possível laço infinito"),
    ("saldo insuficiente na expiração", 'creditar "cliente_42", 1; expirar "cliente_42", 2;', 100, "saldo insuficiente para expirar"),
]
for titulo, fonte, limite, trecho_erro in ERROS_EXECUCAO_DEMONSTRACAO:
    ast, semantica = analisar_semantica(fonte)
    if semantica.erros:
        raise AssertionError(f"O caso '{titulo}' precisa chegar à execução sem erros semânticos.")
    try:
        MaquinaVirtual(limite_instrucoes=limite).executar(gerar_tac(ast))
    except ErroExecucao as erro:
        if trecho_erro not in str(erro):
            raise AssertionError(f"Diagnóstico inesperado para '{titulo}': {erro}") from erro
        print(f"Erro de execução tratado ({titulo}): {erro}")
    else:
        raise AssertionError(f"O caso '{titulo}' deveria falhar na execução.")


Saída da máquina virtual:
  Crédito: cliente_42 recebeu 2 pontos (saldo: 2).
  Resgate: cupom cupom_10 usado por cliente_42; 1 pontos (saldo: 1).
  Expiração: 1 pontos expiraram para cliente_42 (saldo: 0).
  Extrato: cliente_42 possui 0 pontos.
  0
Resumo:
  instrucoes_executadas: 23
  comandos_creditar: 1
  pontos_creditados: 2
  comandos_resgatar: 1
  pontos_resgatados: 1
  comandos_expirar: 1
  pontos_expirados: 1
  comandos_extrato: 1
  clientes_com_saldo: 1
  saldo_total: 0
Erro de execução tratado (divisão por zero): [execução] divisão por zero (PC 11, instrução DIV)
Erro de execução tratado (saldo insuficiente no resgate): [execução] saldo insuficiente para resgatar 3 pontos de cliente_42 (saldo: 2) (PC 1, instrução RESGATAR)
Erro de execução tratado (laço infinito): [execução] limite de 30 instruções atingido; possível laço infinito (PC 0, instrução LABEL)
Erro de execução tratado (saldo insuficiente na expiração): [execução] saldo insuficiente para expirar 2 pontos de cliente_

## Passo 7 — Testes automáticos (R8)

Esta etapa integra as fases em `compilar()` e executa uma bateria de testes de regressão. Cada caso verifica o resultado esperado ou confirma que o erro aparece na fase correta. Os testes semânticos percorrem léxico e sintático antes de falhar; os de execução chegam à VM sem erros de compilação.

In [8]:
# ============================================================
# PASSO 7 - INTEGRACAO E TESTES AUTOMATICOS (R8)
# ============================================================

from dataclasses import dataclass

FASE_VALIDA_R8 = "valido"
FASE_LEXICA_R8 = "l\u00e9xico"
FASE_SINTATICA_R8 = "sint\u00e1tico"
FASE_SEMANTICA_R8 = "sem\u00e2ntico"
FASE_EXECUCAO_R8 = "execu\u00e7\u00e3o"

@dataclass
class ResultadoCompilacao:
    tokens: list
    programa: Programa
    semantica: AnalisadorSemantico
    tac_original: list
    tac_executado: list
    execucao: ResultadoExecucao


def compilar(fonte, otimizar=False, limite_instrucoes=10000):
    """Executa as fases do compilador e retorna o resultado da VM."""
    tokens = tokenizar(fonte)
    programa = analisar_sintaxe(fonte)
    semantica = AnalisadorSemantico()
    semantica.analisar(programa)
    if semantica.erros:
        raise ErroCompilacao(semantica.erros)

    tac_original = gerar_tac(programa)
    tac_executado = otimizar_tac(tac_original) if otimizar else tac_original
    execucao = MaquinaVirtual(limite_instrucoes=limite_instrucoes).executar(tac_executado)
    return ResultadoCompilacao(
        tokens, programa, semantica, tac_original, tac_executado, execucao
    )


CASOS_TESTE_R8 = [
    {
        "nome": "aritmetica e otimizacao",
        "fonte": "inteiro total = 2 + 3; mostrar total;",
        "fase": FASE_VALIDA_R8,
        "saida": ["5"],
        "otimizar": True,
    },
    {
        "nome": "desvio se-senao",
        "fonte": "inteiro pontos = 4; se pontos >= 4 { mostrar 1; } senao { mostrar 0; }",
        "fase": FASE_VALIDA_R8,
        "saida": ["1"],
    },
    {
        "nome": "repeticao enquanto",
        "fonte": "inteiro contador = 2; enquanto contador > 0 { mostrar contador; contador = contador - 1; }",
        "fase": FASE_VALIDA_R8,
        "saida": ["2", "1"],
    },
    {
        "nome": "comandos de fidelidade",
        "fonte": 'creditar "cliente_42", 5; resgatar "cliente_42", "cupom_10", 2; expirar "cliente_42", 1; extrato "cliente_42";',
        "fase": FASE_VALIDA_R8,
        "saida": [
            "Cr\u00e9dito: cliente_42 recebeu 5 pontos (saldo: 5).",
            "Resgate: cupom cupom_10 usado por cliente_42; 2 pontos (saldo: 3).",
            "Expira\u00e7\u00e3o: 1 pontos expiraram para cliente_42 (saldo: 2).",
            "Extrato: cliente_42 possui 2 pontos.",
        ],
        "saldos": {"cliente_42": 2},
        "estatisticas": {
            "comandos_creditar": 1,
            "comandos_resgatar": 1,
            "comandos_expirar": 1,
            "comandos_extrato": 1,
        },
        "otimizar": True,
    },
    {
        "nome": "lexico: caractere inesperado",
        "fonte": "inteiro pontos = 1; @",
        "fase": FASE_LEXICA_R8,
    },
    {
        "nome": "lexico: texto sem fechamento",
        "fonte": 'texto cliente = "sem fechamento;',
        "fase": FASE_LEXICA_R8,
    },
    {
        "nome": "sintatico: expressao ausente",
        "fonte": "inteiro pontos = ;",
        "fase": FASE_SINTATICA_R8,
    },
    {
        "nome": "sintatico: bloco sem fechamento",
        "fonte": "se verdadeiro { mostrar 1;",
        "fase": FASE_SINTATICA_R8,
    },
    {
        "nome": "semantico: variavel nao declarada",
        "fonte": "mostrar saldo_inexistente;",
        "fase": FASE_SEMANTICA_R8,
    },
    {
        "nome": "semantico: tipo incompativel",
        "fonte": "inteiro pontos = 3.5;",
        "fase": FASE_SEMANTICA_R8,
    },
    {
        "nome": "semantico: cashback acima do limite",
        "fonte": "decimal percentual_cashback = 150.0; mostrar percentual_cashback;",
        "fase": FASE_SEMANTICA_R8,
    },
    {
        "nome": "semantico: pontos negativos",
        "fonte": 'creditar "cliente_42", -5;',
        "fase": FASE_SEMANTICA_R8,
    },
    {
        "nome": "execucao: divisao por zero",
        "fonte": "decimal saldo = 1.0; se saldo > 0.0 { saldo = 0.0; } senao { saldo = 1.0; } decimal calculado = 1.0 / saldo; mostrar calculado;",
        "fase": FASE_EXECUCAO_R8,
        "erro": "divis\u00e3o por zero",
    },
    {
        "nome": "execucao: saldo insuficiente no resgate",
        "fonte": 'creditar "cliente_42", 2; resgatar "cliente_42", "cupom_10", 3;',
        "fase": FASE_EXECUCAO_R8,
        "erro": "saldo insuficiente para resgatar",
    },
    {
        "nome": "execucao: saldo insuficiente na expiracao",
        "fonte": 'creditar "cliente_42", 1; expirar "cliente_42", 2;',
        "fase": FASE_EXECUCAO_R8,
        "erro": "saldo insuficiente para expirar",
    },
    {
        "nome": "execucao: limite de instrucoes",
        "fonte": "enquanto verdadeiro { }",
        "fase": FASE_EXECUCAO_R8,
        "limite": 20,
        "erro": "poss\u00edvel la\u00e7o infinito",
    },
]


def executar_testes_r8(casos=CASOS_TESTE_R8):
    resultados = []
    for caso in casos:
        fase_esperada = caso["fase"]
        if fase_esperada == FASE_VALIDA_R8:
            resultado = compilar(
                caso["fonte"],
                otimizar=caso.get("otimizar", False),
                limite_instrucoes=caso.get("limite", 10000),
            )
            if resultado.execucao.saida != caso["saida"]:
                raise AssertionError(
                    f"{caso['nome']}: saida inesperada {resultado.execucao.saida!r}"
                )
            if "saldos" in caso and resultado.execucao.saldos != caso["saldos"]:
                raise AssertionError(
                    f"{caso['nome']}: saldos inesperados {resultado.execucao.saldos!r}"
                )
            for chave, esperado in caso.get("estatisticas", {}).items():
                if resultado.execucao.estatisticas.get(chave) != esperado:
                    raise AssertionError(
                        f"{caso['nome']}: estatistica {chave} inesperada"
                    )
        elif fase_esperada == FASE_EXECUCAO_R8:
            try:
                compilar(
                    caso["fonte"],
                    limite_instrucoes=caso.get("limite", 10000),
                )
            except ErroExecucao as erro:
                if caso["erro"] not in str(erro):
                    raise AssertionError(
                        f"{caso['nome']}: diagnostico de execucao inesperado: {erro}"
                    ) from erro
            else:
                raise AssertionError(f"{caso['nome']}: deveria falhar na execucao")
        else:
            try:
                compilar(caso["fonte"])
            except ErroCompilacao as erro:
                diagnosticos = erro.diagnosticos
                if not diagnosticos or any(
                    diagnostico.fase != fase_esperada for diagnostico in diagnosticos
                ):
                    raise AssertionError(
                        f"{caso['nome']}: falhou na fase errada: {erro}"
                    ) from erro
                if fase_esperada == FASE_LEXICA_R8 and any(
                    diagnostico.linha < 1
                    or diagnostico.coluna < 1
                    or not diagnostico.dica
                    for diagnostico in diagnosticos
                ):
                    raise AssertionError(
                        f"{caso['nome']}: diagnostico lexico sem localizacao ou dica"
                    )
                if fase_esperada == FASE_SINTATICA_R8 and any(
                    not diagnostico.dica for diagnostico in diagnosticos
                ):
                    raise AssertionError(
                        f"{caso['nome']}: diagnostico sintatico sem expectativa"
                    )
            else:
                raise AssertionError(
                    f"{caso['nome']}: deveria falhar na fase {fase_esperada}"
                )
        resultados.append({"nome": caso["nome"], "fase": fase_esperada, "passou": True})

    print("TESTES AUTOMATICOS R8")
    for resultado in resultados:
        print(f"[OK] {resultado['nome']} ({resultado['fase']})")
    print(f"\n{len(resultados)}/{len(casos)} testes passaram.")
    return resultados


RESULTADOS_TESTES_R8 = executar_testes_r8()
if len(RESULTADOS_TESTES_R8) < 12:
    raise AssertionError("R8 exige pelo menos 12 casos de teste")
if sum(item["fase"] == FASE_VALIDA_R8 for item in RESULTADOS_TESTES_R8) < 4:
    raise AssertionError("R8 exige pelo menos 4 programas validos")


TESTES AUTOMATICOS R8
[OK] aritmetica e otimizacao (valido)
[OK] desvio se-senao (valido)
[OK] repeticao enquanto (valido)
[OK] comandos de fidelidade (valido)
[OK] lexico: caractere inesperado (léxico)
[OK] lexico: texto sem fechamento (léxico)
[OK] sintatico: expressao ausente (sintático)
[OK] sintatico: bloco sem fechamento (sintático)
[OK] semantico: variavel nao declarada (semântico)
[OK] semantico: tipo incompativel (semântico)
[OK] semantico: cashback acima do limite (semântico)
[OK] semantico: pontos negativos (semântico)
[OK] execucao: divisao por zero (execução)
[OK] execucao: saldo insuficiente no resgate (execução)
[OK] execucao: saldo insuficiente na expiracao (execução)
[OK] execucao: limite de instrucoes (execução)

16/16 testes passaram.


## Passo 8 — Bateria de testes de regressão (R8)

Além dos 16 casos de integração do Passo 7, esta bateria retesta contratos importantes de R1 a R7, incluindo exemplos da ficha, diagnósticos, AST, escopos, TAC, otimização e estatísticas da VM.

In [9]:
# ============================================================
# PASSO 8 - TESTES DE REGRESSAO DOS REQUISITOS R1 A R7 (R8)
# ============================================================


def exigir_regressao_r8(condicao, mensagem):
    if not condicao:
        raise AssertionError(mensagem)


def testar_ficha_e_exemplos_r8():
    ficha = MINHA_FICHA
    exigir_regressao_r8(
        "TODO" not in ficha["nome"] + ficha["dominio"]
        and len(ficha["tipos"]) >= 3
        and len(ficha["comandos_dominio"]) >= 2
        and len(ficha["regras_de_negocio"]) >= 2
        and len(ficha["programas_exemplo"]) >= 3,
        "A ficha R1 nao atende aos requisitos minimos.",
    )
    for indice, fonte in enumerate(ficha["programas_exemplo"], 1):
        resultado = compilar(fonte)
        exigir_regressao_r8(
            resultado.execucao.estatisticas["instrucoes_executadas"] > 0,
            f"O exemplo R1 {indice} nao executou nenhuma instrucao.",
        )
        tokens = tokenizar(fonte)
        valores = {token.value for token in tokens}
        exigir_regressao_r8(
            "se" in valores or "enquanto" in valores,
            f"O exemplo R1 {indice} nao tem se ou enquanto.",
        )


def testar_lexer_e_comentarios_r8():
    exigir_regressao_r8(
        len(NOMES_TERMINAIS) >= 20 and len(PALAVRAS_RESERVADAS) >= 8,
        "O lexer nao alcanca a quantidade minima de tokens/reservadas.",
    )
    com_comentario = 'inteiro pontos = 3; // comentario ignorado\nmostrar pontos;'
    sem_comentario = 'inteiro pontos = 3;\nmostrar pontos;'
    tokens_com = [(token.type, token.value) for token in tokenizar(com_comentario)]
    tokens_sem = [(token.type, token.value) for token in tokenizar(sem_comentario)]
    exigir_regressao_r8(tokens_com == tokens_sem, "O comentario alterou a sequencia de tokens.")


def testar_dicas_lexicas_r8():
    dicas = [dica_lexica(caractere) for caractere in ("&", "|", "!", "#", "%")]
    exigir_regressao_r8(
        len(set(dicas)) >= 3 and all(dica.strip() for dica in dicas),
        "O lexer deve manter ao menos tres dicas especificas distintas.",
    )


def testar_ast_e_controle_r8():
    fonte = (
        "inteiro contador = 1; "
        "se contador >= 1 { mostrar contador; } "
        "enquanto contador > 0 { contador = contador - 1; }"
    )
    programa = analisar_sintaxe(fonte)
    exigir_regressao_r8(
        is_dataclass(programa)
        and isinstance(programa, Programa)
        and any(isinstance(comando, Se) for comando in programa.comandos)
        and any(isinstance(comando, Enquanto) for comando in programa.comandos),
        "A AST nao representa os nos dataclass de se/enquanto.",
    )


def testar_diagnostico_sintatico_r8():
    try:
        analisar_sintaxe("inteiro pontos = ;")
    except ErroCompilacao as erro:
        diagnosticos = erro.diagnosticos
    else:
        raise AssertionError("O erro sintatico de regressao nao foi detectado.")
    exigir_regressao_r8(
        bool(diagnosticos)
        and all(
            diagnostico.fase == FASE_SINTATICA_R8
            and diagnostico.linha >= 1
            and diagnostico.coluna >= 1
            and bool(diagnostico.dica)
            for diagnostico in diagnosticos
        ),
        "O diagnostico sintatico perdeu fase, localizacao ou expectativa.",
    )


def testar_coercao_semantica_r8():
    programa, semantica = analisar_semantica(PROGRAMA_COERCAO)
    exigir_regressao_r8(
        not semantica.erros and isinstance(programa.comandos[0].expr, Conv),
        "A conversao inteiro para decimal nao foi registrada na AST.",
    )


def testar_escopo_semantico_r8():
    _, semantica = analisar_semantica(
        "se verdadeiro { inteiro bonus = 10; mostrar bonus; } mostrar bonus;"
    )
    exigir_regressao_r8(
        bool(semantica.erros)
        and any("bonus" in diagnostico.mensagem for diagnostico in semantica.erros),
        "Uma variavel local escapou do escopo de bloco.",
    )


def testar_avisos_semanticos_r8():
    resultado = compilar(PROGRAMA_AVISOS)
    exigir_regressao_r8(
        not resultado.semantica.erros and bool(resultado.semantica.avisos),
        "Avisos semanticos devem ser preservados sem impedir a execucao.",
    )


def testar_galeria_semantica_r8():
    for titulo, fonte in GALERIA_SEMANTICA:
        tokenizar(fonte)
        programa = analisar_sintaxe(fonte)
        analisador = AnalisadorSemantico()
        analisador.analisar(programa)
        exigir_regressao_r8(
            bool(analisador.erros),
            f"O caso '{titulo}' deixou de produzir erro semantico.",
        )


def testar_tac_e_desvios_r8():
    programa = analisar_sintaxe(PROGRAMA_TAC)
    operacoes = {instrucao.op for instrucao in gerar_tac(programa)}
    exigir_regressao_r8(
        {"CREDITAR", "RESGATAR", "EXPIRAR", "EXTRATO", "LABEL", "GOTO"} <= operacoes,
        "O TAC nao contem todas as instrucoes de dominio e controle.",
    )
    condicional = analisar_sintaxe(
        "inteiro pontos = 1; se pontos >= 1 { mostrar pontos; }"
    )
    desvios = [
        instrucao.op
        for instrucao in gerar_tac(condicional)
        if instrucao.op.startswith("IF_")
    ]
    exigir_regressao_r8(
        "IF_LT" in desvios,
        "A condicao >= deveria gerar desvio invertido com IF_LT.",
    )


def testar_otimizacao_equivalente_r8():
    for comparacao in RESULTADOS_OTIMIZACAO:
        antes = MaquinaVirtual().executar(comparacao["tac_antes"])
        depois = MaquinaVirtual().executar(comparacao["tac_depois"])
        exigir_regressao_r8(
            antes.saida == depois.saida
            and depois.estatisticas["instrucoes_executadas"]
            < antes.estatisticas["instrucoes_executadas"],
            f"A otimizacao nao preservou a saida/reduziu instrucoes em {comparacao['nome']}.",
        )


def testar_estatisticas_vm_r8():
    caso_dominio = next(
        caso for caso in CASOS_TESTE_R8 if caso["nome"] == "comandos de fidelidade"
    )
    resultado = compilar(caso_dominio["fonte"], otimizar=True)
    comandos = ("creditar", "resgatar", "expirar", "extrato")
    exigir_regressao_r8(
        all(resultado.execucao.estatisticas[f"comandos_{comando}"] == 1 for comando in comandos)
        and len(resultado.execucao.saida) == 4
        and resultado.execucao.saldos == {"cliente_42": 2},
        "A VM nao preservou saida, saldo ou estatisticas dos comandos de dominio.",
    )


def testar_percentual_cashback_dinamico_r8():
    fonte = '''decimal limite = 0.0; logico flag = verdadeiro;
se flag { limite = 150.0; } senao { limite = 0.0; }
decimal percentual_cashback = limite; mostrar percentual_cashback;'''
    _, semantica = analisar_semantica(fonte)
    exigir_regressao_r8(
        not semantica.erros,
        "O caso precisa passar pela análise semantica para exercitar a VM.",
    )
    for otimizar in (False, True):
        try:
            compilar(fonte, otimizar=otimizar)
        except ErroExecucao as erro:
            exigir_regressao_r8(
                "percentual de cashback inválido" in str(erro)
                and "limite é 100%" in str(erro),
                "A VM nao identificou o percentual dinamico acima do limite.",
            )
        except ErroCompilacao as erro:
            raise AssertionError(
                "O caso deve ser detectado em execucao, nao na compilacao."
            ) from erro
        else:
            raise AssertionError(
                "A VM aceitou cashback dinamico acima de 100%."
            )


TESTES_REGRESSAO_R8 = [
    ("R1 - ficha e exemplos", testar_ficha_e_exemplos_r8),
    ("R2 - tokens e comentarios", testar_lexer_e_comentarios_r8),
    ("R2 - dicas lexicas", testar_dicas_lexicas_r8),
    ("R3 - AST e controle", testar_ast_e_controle_r8),
    ("R3 - diagnostico sintatico", testar_diagnostico_sintatico_r8),
    ("R4 - coercao", testar_coercao_semantica_r8),
    ("R4 - escopo", testar_escopo_semantico_r8),
    ("R4 - avisos", testar_avisos_semanticos_r8),
    ("R4 - galeria semantica", testar_galeria_semantica_r8),
    ("R5 - TAC e desvios", testar_tac_e_desvios_r8),
    ("R6 - otimizacao equivalente", testar_otimizacao_equivalente_r8),
    ("R7 - estatisticas da VM", testar_estatisticas_vm_r8),
    ("R7 - cashback dinamico", testar_percentual_cashback_dinamico_r8),
]

RESULTADOS_REGRESSAO_R8 = []
for nome, teste in TESTES_REGRESSAO_R8:
    teste()
    RESULTADOS_REGRESSAO_R8.append({"nome": nome, "passou": True})
    print(f"[OK] {nome}")

TOTAL_TESTES_R8 = len(RESULTADOS_TESTES_R8) + len(RESULTADOS_REGRESSAO_R8)
if len(RESULTADOS_REGRESSAO_R8) < 12 or TOTAL_TESTES_R8 < 12:
    raise AssertionError("R8 exige pelo menos 12 testes automatizados.")
print(f"\n{len(RESULTADOS_REGRESSAO_R8)}/{len(TESTES_REGRESSAO_R8)} testes de regressao passaram.")
print(f"Total integrado R8: {TOTAL_TESTES_R8} casos/testes.")


[OK] R1 - ficha e exemplos
[OK] R2 - tokens e comentarios
[OK] R2 - dicas lexicas
[OK] R3 - AST e controle
[OK] R3 - diagnostico sintatico
[OK] R4 - coercao
[OK] R4 - escopo
[OK] R4 - avisos
[OK] R4 - galeria semantica
[OK] R5 - TAC e desvios
[OK] R6 - otimizacao equivalente
[OK] R7 - estatisticas da VM
[OK] R7 - cashback dinamico

13/13 testes de regressao passaram.
Total integrado R8: 29 casos/testes.


## Passo 9 — FidelidadeScript Studio (R9)

Interface interativa em `ipywidgets` para selecionar exemplos ou editar um programa e inspecionar sua execucao, tokens, TAC e AST. O botao de compilacao mostra diagnosticos de compilacao e execucao no painel, sem ocultar erros inesperados.

In [10]:
# ============================================================
# PASSO 9 - INTERFACE FIDELITYSCRIPT STUDIO (R9)
# ============================================================

from contextlib import redirect_stdout
from html import escape as escapar_html
from io import StringIO
from IPython.display import HTML, display
import ipywidgets as widgets

EXEMPLOS_INTERFACE = [
    ("Campanha de cashback", PROGRAMA_FIDELIDADE),
    ("Fluxo de execução", PROGRAMA_EXECUCAO),
    ("Decisao se/senao", CASOS_TESTE_R8[1]["fonte"]),
    ("Repeticao enquanto", CASOS_TESTE_R8[2]["fonte"]),
]

ESTADO_INTERFACE = {"ultimo_resultado": None, "ultimo_erro": None}

_ESTILO_INTERFACE = """
<style>
.fidelidade { font-family: Segoe UI, sans-serif; color: #172033; }
.fidelidade h3 { margin: 0 0 8px; color: #173b73; }
.fidelidade .subtitulo { color: #526078; margin-bottom: 12px; }
.fidelidade pre { white-space: pre-wrap; overflow-wrap: anywhere; background: #f4f6fa;
  border: 1px solid #dfe5ef; border-radius: 6px; padding: 12px; margin: 8px 0; }
.fidelidade table { border-collapse: collapse; width: 100%; margin: 8px 0; }
.fidelidade th, .fidelidade td { border: 1px solid #dfe5ef; padding: 6px 8px; text-align: left; }
.fidelidade th { background: #edf2fa; }
.fidelidade .erro { color: #8b1e2d; background: #fff1f2; border-left: 4px solid #c6283d;
  padding: 10px; white-space: pre-wrap; }
.fidelidade .aviso { color: #684b00; background: #fff8df; border-left: 4px solid #d5a500;
  padding: 8px; margin: 6px 0; }
.fidelidade .ok { color: #17613a; font-weight: 600; }
</style>
"""


def _bloco_html(titulo, conteudo, classe=""):
    classe_html = f" class=\"{escapar_html(classe, quote=True)}\"" if classe else ""
    return (
        _ESTILO_INTERFACE
        + '<section class="fidelidade">'
        + f"<h3>{escapar_html(str(titulo))}</h3>"
        + f"<div{classe_html}>{conteudo}</div></section>"
    )


def _pre_html(texto):
    return f"<pre>{escapar_html(str(texto))}</pre>"


def _tabela_html(cabecalhos, linhas):
    th = "".join(f"<th>{escapar_html(str(item))}</th>" for item in cabecalhos)
    trs = []
    for linha in linhas:
        tds = "".join(f"<td>{escapar_html(str(item))}</td>" for item in linha)
        trs.append(f"<tr>{tds}</tr>")
    return f"<table><thead><tr>{th}</tr></thead><tbody>{''.join(trs)}</tbody></table>"


def _renderizar_tokens_interface(tokens):
    linhas = [
        (indice, token.type, token.value, f"{token.line}:{token.column}")
        for indice, token in enumerate(tokens, 1)
    ]
    return _bloco_html(
        f"Tokens ({len(tokens)})",
        _tabela_html(("#", "Tipo", "Lexema", "Linha:coluna"), linhas),
    )


def _renderizar_tac_interface(resultado):
    original = "\\n".join(formatar_instrucao_tac(item) for item in resultado.tac_original)
    executado = "\\n".join(formatar_instrucao_tac(item) for item in resultado.tac_executado)
    if not original:
        original = "(vazio)"
    if not executado:
        executado = "(vazio)"
    titulo_exec = "TAC otimizado" if resultado.tac_executado is not resultado.tac_original else "TAC executado"
    return _bloco_html(
        "Código de Três Endereços",
        f"<h4>Original ({len(resultado.tac_original)} instruções)</h4>"
        + _pre_html(original)
        + f"<h4>{escapar_html(titulo_exec)} ({len(resultado.tac_executado)} instruções)</h4>"
        + _pre_html(executado),
    )


def _renderizar_ast_interface(programa):
    buffer = StringIO()
    with redirect_stdout(buffer):
        mostrar_ast(programa)
    return _bloco_html("Árvore sintática abstrata (AST)", _pre_html(buffer.getvalue()))


def _renderizar_execucao_interface(resultado):
    saida = "".join(f"<li>{escapar_html(str(item))}</li>" for item in resultado.execucao.saida)
    if not saida:
        saida = "<li>(sem saída)</li>"
    estatisticas = _tabela_html(
        ("Estatística", "Valor"),
        sorted(resultado.execucao.estatisticas.items()),
    )
    avisos = "".join(
        f'<div class="aviso">{escapar_html(str(aviso))}</div>'
        for aviso in resultado.semantica.avisos
    )
    resumo = (
        "<p class=\"ok\">Compilação e execução concluídas.</p>"
        "<h4>Saída</h4><ul>" + saida + "</ul>"
        + "<h4>Saldos por cliente</h4>" + _pre_html(repr(resultado.execucao.saldos))
        + "<h4>Estatísticas da máquina virtual</h4>" + estatisticas
        + ("<h4>Avisos</h4>" + avisos if avisos else "")
    )
    return _bloco_html("Execução", resumo)


def _renderizar_erro_interface(erro):
    if isinstance(erro, ErroCompilacao):
        titulo = "Erro de compilacao"
    elif isinstance(erro, ErroExecucao):
        titulo = "Erro de execução"
    else:
        raise TypeError("Tipo de erro inesperado na interface")
    return _bloco_html(titulo, _pre_html(str(erro)), "erro")


def _mostrar_interface(output, conteudo):
    with output:
        output.clear_output(wait=True)
        display(HTML(conteudo))


def _mostrar_visao_indisponivel(output, nome_visao):
    mensagem = (
        f"A visão {nome_visao} só fica disponível quando o programa compila e executa. "
        "Consulte o diagnóstico na aba Execução."
    )
    _mostrar_interface(output, _bloco_html(nome_visao, escapar_html(mensagem)))


seletor_exemplos = widgets.Dropdown(
    options=EXEMPLOS_INTERFACE,
    value=EXEMPLOS_INTERFACE[0][1],
    description="Exemplo:",
    layout=widgets.Layout(width="min(100%, 760px)"),
    style={"description_width": "initial"},
)
editor_fidelidade = widgets.Textarea(
    value=EXEMPLOS_INTERFACE[0][1],
    description="Programa:",
    placeholder="Escreva um programa FidelidadeScript...",
    layout=widgets.Layout(width="100%", height="280px"),
    style={"description_width": "initial"},
)
otimizar_interface = widgets.Checkbox(
    value=False,
    description="Executar TAC otimizado",
    indent=False,
)
botao_executar_interface = widgets.Button(
    description="Compilar e executar",
    tooltip="Executar todas as fases e atualizar as visões",
    icon="play",
    button_style="success",
)
estado_interface_html = widgets.HTML(
    value="<span>Escolha um exemplo ou edite o programa e execute.</span>"
)

saida_execucao_interface = widgets.Output(layout=widgets.Layout(width="100%"))
saida_tokens_interface = widgets.Output(layout=widgets.Layout(width="100%"))
saida_tac_interface = widgets.Output(layout=widgets.Layout(width="100%"))
saida_ast_interface = widgets.Output(layout=widgets.Layout(width="100%"))
abas_interface = widgets.Tab(
    children=(
        saida_execucao_interface,
        saida_tokens_interface,
        saida_tac_interface,
        saida_ast_interface,
    ),
    layout=widgets.Layout(width="100%"),
)
for indice, titulo in enumerate(("Execução", "Tokens", "TAC", "AST")):
    abas_interface.set_title(indice, titulo)


def _carregar_exemplo_interface(evento):
    if evento.get("name") == "value" and evento.get("new") is not None:
        editor_fidelidade.value = evento["new"]
        estado_interface_html.value = "<span>Exemplo carregado. Clique em Compilar e executar.</span>"


def executar_interface_fidelidade(evento=None):
    fonte = editor_fidelidade.value
    estado_interface_html.value = "<span>Compilando e executando...</span>"
    ESTADO_INTERFACE["ultimo_resultado"] = None
    ESTADO_INTERFACE["ultimo_erro"] = None
    try:
        resultado = compilar(fonte, otimizar=otimizar_interface.value)
    except (ErroCompilacao, ErroExecucao) as erro:
        ESTADO_INTERFACE["ultimo_erro"] = erro
        _mostrar_interface(saida_execucao_interface, _renderizar_erro_interface(erro))
        _mostrar_visao_indisponivel(saida_tokens_interface, "Tokens")
        _mostrar_visao_indisponivel(saida_tac_interface, "TAC")
        _mostrar_visao_indisponivel(saida_ast_interface, "AST")
        estado_interface_html.value = (
            "<span style=\"color:#8b1e2d;font-weight:600\">Programa não concluído; consulte o diagnóstico.</span>"
        )
        return

    ESTADO_INTERFACE["ultimo_resultado"] = resultado
    _mostrar_interface(saida_execucao_interface, _renderizar_execucao_interface(resultado))
    _mostrar_interface(saida_tokens_interface, _renderizar_tokens_interface(resultado.tokens))
    _mostrar_interface(saida_tac_interface, _renderizar_tac_interface(resultado))
    _mostrar_interface(saida_ast_interface, _renderizar_ast_interface(resultado.programa))
    estado_interface_html.value = (
        "<span style=\"color:#17613a;font-weight:600\">Execução concluída com sucesso.</span>"
    )


seletor_exemplos.observe(_carregar_exemplo_interface, names="value")
botao_executar_interface.on_click(executar_interface_fidelidade)

cabecalho_interface = widgets.HTML(
    "<div style='padding:14px 16px;background:#edf2fa;border-radius:8px;margin-bottom:10px'>"
    "<h2 style='margin:0;color:#173b73'>FidelidadeScript Studio</h2>"
    "<p style='margin:6px 0 0;color:#526078'>Editor e painel de inspeção para a DSL de pontos e cashback.</p>"
    "</div>"
)
painel_fidelidade = widgets.VBox(
    (
        cabecalho_interface,
        seletor_exemplos,
        editor_fidelidade,
        widgets.HBox((botao_executar_interface, otimizar_interface)),
        estado_interface_html,
        abas_interface,
    ),
    layout=widgets.Layout(width="100%", max_width="1100px", gap="10px"),
)

display(painel_fidelidade)
